# Actividad 04 · Tres formas de recombinar y tres de mutar en un algoritmo genético para fijar precios y descuentos

| | |
|---|---|
| **Alumno** | Víctor Manuel Santos Martínez — matrícula 253220020 |
| **Materia** | Algoritmos Evolutivos — Maestría en Inteligencia Artificial |
| **Docente** | Dr. Jaime Aguilar Ortiz |
| **Actividad** | Actividad 04 — Comparación de tres formas de realizar dos de las siete etapas de un algoritmo genético |
| **Modalidad** | Individual |
| **Periodo** | Septiembre–diciembre de 2026 · Corte 2 |
| **Institución** | Universidad Politécnica Metropolitana de Hidalgo |

**Consigna.** *Desarrollar una aplicación de algoritmo genético aplicando y comparando tres formas de realizar por lo menos dos de las 7 etapas de un algoritmo genético y justificar la elección.* El informe debe describir el problema, definir el objetivo del estudio, definir los métodos de las siete etapas, generar un código que apoye el objetivo, interpretar los resultados y anexar referencias.

**Qué hace este cuadernillo.** Formula un problema empresarial con genes reales —una cadena de tiendas fija el precio de temporada de cada producto de una categoría, con pisos, techos, existencias limitadas, una escalera de precios entre marcas y un tope al índice de precios de la categoría— y lo resuelve con un algoritmo genético de siete etapas en el que **dos etapas se realizan de tres formas distintas**:

| Etapa | Formas comparadas | Funciones |
|---|---|---|
| 4. Recombinación | **M14** uniforme por máscara · **M15** aritmética · **M16** BLX-α | `etapa_4_recombinar_M14`, `_M15`, `_M16` |
| 5. Mutación | **M18** reinicio uniforme · **M19** gaussiana acotada · **M20** polinomial acotada | `etapa_5_mutar_M18`, `_M19`, `_M20` |

Las otras cinco etapas quedan fijas: **M03** hipercubo latino, **M08** prioridad de factibilidad, **M12** torneo, **M22** reemplazo elitista y **M26** presupuesto de evaluaciones. Las nueve combinaciones se comparan en un **diseño factorial completo 3 × 3**, con los parámetros propios de cada operador calibrados en una instancia aparte, treinta semillas emparejadas por instancia y cuatro instancias de evaluación cuyo óptimo global se conoce y se certifica.

**Orden de lectura.** A, entorno · B, modelo · C, óptimo exacto y líneas base · D, las siete etapas · E, pruebas · F, instancias · G, diseño experimental e hipótesis · H, calibración · I, experimento factorial · J, análisis estadístico · K, mecanismos · L, caso principal · M, figuras · N, manifiesto · O, lectura de resultados.

Todos los datos son **sintéticos**: no proceden de una empresa real. Los precios están en pesos y la demanda en unidades por temporada de cuatro semanas; son una escala de trabajo, no una estimación comercial.

## A. Entorno y dependencias

### A.1 Comprobación del entorno

La primera celda falla de inmediato si el kernel no pertenece al entorno conda `science`. Se comprueba la ruta del intérprete que ejecuta el código, no el nombre visible del kernel.

In [1]:
import sys
import platform
from pathlib import Path

if "envs/science" not in sys.executable.replace("\\", "/"):
    raise RuntimeError(
        f"Este cuadernillo debe ejecutarse con el entorno conda 'science'. Intérprete actual: {sys.executable}"
    )
print("Intérprete:", sys.executable)
print("Python", platform.python_version(), "·", platform.platform())
print("Directorio de trabajo:", Path.cwd())

Intérprete: /opt/homebrew/Caskroom/miniconda/base/envs/science/bin/python3.11
Python 3.11.16 · macOS-27.0.1-arm64-arm-64bit
Directorio de trabajo: /Users/victormanuel/Library/Mobile Documents/com~apple~CloudDocs/Maestría UMPH/Cuatrimestre 4/Algoritmos Evolutivos/Actividades/Actividad 4


### A.2 Dependencias, constantes y estilo de las figuras

Sólo bibliotecas generales: NumPy para el cálculo vectorizado, pandas para las tablas, Matplotlib para las figuras y SciPy para tres tareas que **no** forman parte del algoritmo genético: el optimizador de programación cuadrática secuencial que da el punto de partida de la solución exacta (`scipy.optimize.minimize` con SLSQP), la prueba de Friedman y la de rangos con signo de Wilcoxon. No se usa ninguna biblioteca especializada en algoritmos genéticos.

La codificación visual es fija en todas las figuras: **el color identifica la recombinación** (M14 azul, M15 naranja, M16 aqua) y **el trazo y el marcador identifican la mutación** (M18 punteado con círculo, M19 discontinuo con cuadrado, M20 continuo con triángulo). Así ninguna configuración depende sólo del color.

In [2]:
import hashlib
import json
import time
from dataclasses import dataclass, replace
from itertools import product

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle
from scipy.optimize import minimize
from scipy.stats import friedmanchisquare, norm, rankdata, wilcoxon

RESULTADOS = Path("resultados")
FIGURAS = Path("figuras")
RESULTADOS.mkdir(exist_ok=True)
FIGURAS.mkdir(exist_ok=True)
INICIO_CUADERNILLO = time.perf_counter()

CRUCES = ("M14", "M15", "M16")
MUTACIONES = ("M18", "M19", "M20")
CONFIGURACIONES = [(c, m) for c in CRUCES for m in MUTACIONES]
NOMBRE = {"M14": "M14 uniforme", "M15": "M15 aritmética", "M16": "M16 BLX-α",
          "M18": "M18 reinicio", "M19": "M19 gaussiana", "M20": "M20 polinomial"}

# Paleta categórica validada (azul, naranja, aqua), tinta y rampa secuencial azul.
AZUL, NARANJA, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
TINTA, TINTA_2, TENUE = "#0b0b0b", "#52514e", "#898781"
REJILLA, EJE, GRIS_CLARO = "#e1e0d9", "#c3c2b7", "#d9d8d2"
RAMPA_AZUL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
COLOR_CRUCE = {"M14": AZUL, "M15": NARANJA, "M16": AQUA}
TRAZO_MUTACION = {"M18": ":", "M19": "--", "M20": "-"}
MARCA_MUTACION = {"M18": "o", "M19": "s", "M20": "^"}
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "figure.facecolor": "white", "axes.facecolor": "white",
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.edgecolor": EJE, "axes.linewidth": 0.8, "axes.labelcolor": TINTA_2,
    "axes.titlecolor": TINTA, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.6, "grid.linestyle": "-",
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2.0,
})
print("NumPy", np.__version__, "· pandas", pd.__version__, "· SciPy", scipy.__version__,
      "· Matplotlib", matplotlib.__version__)

NumPy 2.4.6 · pandas 3.0.6 · SciPy 1.17.1 · Matplotlib 3.11.2


## B. Modelo del problema

### B.1 La decisión y sus reglas

Una cadena regional de tiendas de conveniencia fija el **precio de temporada** de los $d$ productos de una categoría. Los productos se agrupan en familias —por ejemplo, agua natural de 1.5 L— y cada familia tiene tres niveles: marca propia, marca estándar y marca premium. Hoy los precios salen de una regla de **costo más 45 %**, igual para todos, que ignora cuánto reacciona cada producto a su precio. El gen $j$ es el precio $p_j$ en pesos: un número real.

**Demanda.** La demanda de la temporada es lineal en los precios, con efectos cruzados entre sustitutos:

$$q(p)=a-Bp,\qquad B_{jj}=b_j>0,\qquad B_{jk}=-\kappa_{jk}\sqrt{b_jb_k}\le 0\ (j\neq k).$$

La pendiente propia sale de la elasticidad $\varepsilon_j$ medida al precio actual $P_j$, $b_j=\varepsilon_jq^0_j/P_j$, y el intercepto se fija para que $q(P)=q^0$. Un descuento en un producto le quita ventas a sus sustitutos; dentro de una familia el efecto cruzado es mayor que entre familias. $B$ es simétrica y definida positiva, de modo que la utilidad

$$\pi(p)=\sum_j (p_j-c_j)\,q_j(p)=-p^\top Bp+(a+Bc)^\top p-c^\top a$$

es **estrictamente cóncava**. Esa propiedad no la usa el algoritmo genético; la usa la referencia exacta de la sección C para certificar el óptimo global.

**Reglas comerciales.** Además del intervalo $[\ell_j,u_j]$ de cada precio —piso de margen mínimo y techo de alza máxima—, hay cuatro familias de restricciones lineales, escritas como $Gp\le h$ con cada fila adimensional:

| Regla | Forma | Escala de la fila |
|---|---|---|
| Existencias | $q_j(p)\le S_j$ | $S_j$ |
| Demanda no negativa | $q_j(p)\ge 0$ | $q^0_j$ |
| Escalera de precios | $p_{\text{superior}}\ge(1+g)\,p_{\text{inferior}}$ dentro de cada familia | $P_{\text{superior}}$ |
| Índice de precios | $\sum_j w_j\,p_j/P_j\le 1+\eta_I$, con $w_j$ la participación en el ingreso actual | 1 |

La **violación agregada** $V(p)=\sum_i\max\{0,(Gp-h)_i\}$ vale cero exactamente en el conjunto factible, que es un **poliedro convexo**: la intersección de semiespacios con la caja de precios.

In [3]:
NIVELES = ("marca propia", "marca estándar", "marca premium")


@dataclass(frozen=True)
class Instancia:
    """Un caso de fijación de precios. Precios en pesos; demanda en unidades por temporada."""
    nombre: str
    familias: tuple
    familia: np.ndarray        # familia de cada producto
    nivel: np.ndarray          # 0 propia, 1 estándar, 2 premium
    costo: np.ndarray          # c_j
    precio_actual: np.ndarray  # P_j, regla vigente de costo más 45 %
    demanda_actual: np.ndarray # q0_j = q_j(P)
    B: np.ndarray              # pendientes: simétrica y definida positiva
    a: np.ndarray              # intercepto: q(p) = a - B p
    existencias: np.ndarray    # S_j
    piso: np.ndarray           # l_j
    techo: np.ndarray          # u_j
    G: np.ndarray              # reglas comerciales: G p <= h
    h: np.ndarray
    tipo: tuple                # (regla, producto) de cada fila de G
    brecha_escalera: float
    holgura_indice: float
    pesos_indice: np.ndarray

    @property
    def d(self) -> int:
        return len(self.costo)

    def producto(self, j: int) -> str:
        return f"{self.familias[self.familia[j]]} · {NIVELES[self.nivel[j]]}"

    def demanda(self, X):
        return self.a - np.asarray(X) @ self.B


def ensamblar_instancia(nombre, familias, familia, nivel, costo, precio_actual, demanda_actual, B,
                        existencias, piso, techo, brecha_escalera, holgura_indice) -> Instancia:
    """Calcula el intercepto de la demanda y escribe las reglas comerciales como G p <= h."""
    familia, nivel = np.asarray(familia, int), np.asarray(nivel, int)
    B, P = np.asarray(B, float), np.asarray(precio_actual, float)
    q0, S = np.asarray(demanda_actual, float), np.asarray(existencias, float)
    a = q0 + B @ P
    d = len(P)
    filas, lados, tipo = [], [], []
    for j in range(d):                      # existencias: a_j - B_j p <= S_j
        filas.append(-B[j] / S[j]); lados.append((S[j] - a[j]) / S[j]); tipo.append(("existencias", j))
    for j in range(d):                      # demanda no negativa: B_j p - a_j <= 0
        filas.append(B[j] / q0[j]); lados.append(a[j] / q0[j]); tipo.append(("demanda", j))
    for f in np.unique(familia):            # escalera: (1 + g) p_inferior - p_superior <= 0
        miembros = np.flatnonzero(familia == f)
        miembros = miembros[np.argsort(nivel[miembros], kind="stable")]
        for i, s in zip(miembros[:-1], miembros[1:]):
            fila = np.zeros(d); fila[i] = 1.0 + brecha_escalera; fila[s] = -1.0
            filas.append(fila / P[s]); lados.append(0.0); tipo.append(("escalera", int(s)))
    w = q0 * P / np.sum(q0 * P)             # índice de precios ponderado por el ingreso actual
    filas.append(w / P); lados.append(1.0 + holgura_indice); tipo.append(("índice", -1))
    return Instancia(nombre, tuple(familias), familia, nivel, np.asarray(costo, float), P, q0, B, a, S,
                     np.asarray(piso, float), np.asarray(techo, float), np.array(filas), np.array(lados),
                     tuple(tipo), float(brecha_escalera), float(holgura_indice), w)


def validar_instancia(inst: Instancia) -> None:
    """Rechaza datos incoherentes; no corrige nada en silencio."""
    d = inst.d
    if inst.B.shape != (d, d) or not np.allclose(inst.B, inst.B.T, rtol=0, atol=1e-12):
        raise ValueError(f"{inst.nombre}: B debe ser simétrica de {d}×{d}.")
    if np.linalg.eigvalsh(inst.B).min() <= 0:
        raise ValueError(f"{inst.nombre}: B no es definida positiva; la utilidad no sería estrictamente cóncava.")
    if not (np.all(inst.costo < inst.piso) and np.all(inst.piso < inst.precio_actual)
            and np.all(inst.precio_actual <= inst.techo)):
        raise ValueError(f"{inst.nombre}: se requiere costo < piso < precio actual <= techo.")
    if not np.allclose(inst.demanda(inst.precio_actual), inst.demanda_actual, rtol=1e-12, atol=1e-9):
        raise ValueError(f"{inst.nombre}: la demanda al precio actual no coincide con q0.")
    if np.any(inst.G @ inst.precio_actual > inst.h + 1e-12):
        raise ValueError(f"{inst.nombre}: los precios actuales deben cumplir las reglas comerciales.")

### B.2 Generador de instancias sintéticas

Cada instancia se genera con una semilla. El costo de cada familia se sortea, se multiplica por 0.70, 1.00 y 1.40 según el nivel, y el precio actual es el costo más 45 %. La demanda actual reparte el volumen de la familia en 35 %, 45 % y 20 %. Las elasticidades propias siguen el patrón habitual del comercio minorista: la marca propia es la más sensible al precio y la premium la menos.

| Parámetro | Valor | Significado |
|---|---|---|
| Piso $\ell_j$ | $1.10\,c_j$ | Margen mínimo de 10 % sobre el costo |
| Techo $u_j$ | $1.20\,P_j$ | El precio no sube más de 20 % en la temporada |
| Escalera $g$ | 8 % | Cada nivel cuesta al menos 8 % más que el inferior de su familia |
| Índice $\eta_I$ | 3 % | El índice de precios de la categoría sube a lo más 3 % |
| Elasticidad propia | $[3.0,5.5]$, $[2.0,3.8]$, $[1.3,2.4]$ | Propia, estándar y premium |
| Existencias $S_j$ | $q^0_j\times\mathcal U(1.05,1.50)$ | Lo que el proveedor puede surtir en la temporada |

La **intensidad de sustitución** distingue dos tipos de instancia. Con sustitución **débil (D)**, cada par de una familia tiene $\kappa=0.10$ y la suma de los efectos entre familias es 0.05. Con sustitución **fuerte (F)**, $\kappa=0.30$ y 0.12. La sustitución fuerte acopla los precios de cada familia: la utilidad casi no cambia si los tres niveles se mueven juntos, pero cae rápido si se separan. Es un valle largo y estrecho en el espacio de los genes, y es ahí donde la forma de recombinar debería importar más.

In [4]:
MARGEN_ACTUAL = 0.45          # regla vigente: precio = costo × 1.45
MARGEN_MINIMO = 0.10          # piso: costo × 1.10
ALZA_MAXIMA = 0.20            # techo: precio actual × 1.20
BRECHA_ESCALERA = 0.08
HOLGURA_INDICE = 0.03
MULTIPLICADOR_COSTO = np.array([0.70, 1.00, 1.40])
CUOTA_NIVEL = np.array([0.35, 0.45, 0.20])
ELASTICIDAD = ((3.0, 5.5), (2.0, 3.8), (1.3, 2.4))
SUSTITUCION = {"D": (0.10, 0.05), "F": (0.30, 0.12)}   # (κ por par en la familia, suma entre familias)
EXISTENCIAS = (1.05, 1.50)


def generar_instancia(nombre: str, n_familias: int, intensidad: str, semilla: int,
                      familias=None, costos_base=None, volumenes=None) -> Instancia:
    """Genera una categoría de n_familias × 3 productos con la intensidad de sustitución dada."""
    if n_familias < 2 or intensidad not in SUSTITUCION:
        raise ValueError("Se requieren al menos dos familias y una intensidad 'D' o 'F'.")
    rng = np.random.default_rng(semilla)
    F, d = n_familias, 3 * n_familias
    familia, nivel = np.repeat(np.arange(F), 3), np.tile(np.arange(3), F)
    familias = tuple(familias) if familias is not None else tuple(f"Familia {f + 1:02d}" for f in range(F))
    base = np.asarray(costos_base, float) if costos_base is not None else rng.uniform(8, 30, F)
    volumen = np.asarray(volumenes, float) if volumenes is not None else rng.uniform(800, 3500, F)
    costo = np.round(base[familia] * MULTIPLICADOR_COSTO[nivel] * rng.uniform(0.95, 1.05, d), 2)
    precio = np.round(costo * (1 + MARGEN_ACTUAL), 1)
    q0 = np.round(volumen[familia] * CUOTA_NIVEL[nivel] * rng.uniform(0.85, 1.15, d))
    elasticidad = np.array([rng.uniform(*ELASTICIDAD[t]) for t in nivel])
    b = elasticidad * q0 / precio
    kappa, entre = SUSTITUCION[intensidad]
    K = np.where(familia[:, None] == familia[None, :], kappa, entre / (d - 3))
    np.fill_diagonal(K, 0.0)
    raiz = np.sqrt(b)
    B = np.diag(b) - K * np.outer(raiz, raiz)
    existencias = np.round(q0 * rng.uniform(*EXISTENCIAS, d))
    piso = np.round(costo * (1 + MARGEN_MINIMO), 2)
    techo = np.round(precio * (1 + ALZA_MAXIMA), 2)
    inst = ensamblar_instancia(nombre, familias, familia, nivel, costo, precio, q0, B, existencias,
                               piso, techo, BRECHA_ESCALERA, HOLGURA_INDICE)
    validar_instancia(inst)
    return inst

### B.3 Evaluación vectorizada y evaluación escalar independiente

`utilidad` y `violacion` evalúan una población completa con dos productos de matrices. `evaluar_escalar` recalcula lo mismo para un solo vector de precios **con bucles y desde el significado de cada regla**, sin usar $G$ ni $h$. Sirve para auditar la evaluación vectorizada en las pruebas y la solución que entrega cada corrida.

In [5]:
TOL_FACTIBLE = 1e-12   # unidades relativas de la violación; absorbe sólo el redondeo de punto flotante


def utilidad(inst: Instancia, X) -> np.ndarray:
    """Utilidad de la temporada de cada fila de X (pesos)."""
    X = np.atleast_2d(X)
    return np.sum((X - inst.costo) * (inst.a - X @ inst.B), axis=1)


def violacion(inst: Instancia, X) -> np.ndarray:
    """Violación agregada V de cada fila de X; vale 0 exactamente en el conjunto factible."""
    X = np.atleast_2d(X)
    return np.maximum(0.0, X @ inst.G.T - inst.h).sum(axis=1)


def evaluar_escalar(inst: Instancia, p) -> dict:
    """Recalcula utilidad y reglas con bucles, a partir de su significado y sin usar G ni h."""
    d = inst.d
    p = [float(v) for v in p]
    q = [inst.a[j] - sum(inst.B[j, k] * p[k] for k in range(d)) for j in range(d)]
    valor = sum((p[j] - inst.costo[j]) * q[j] for j in range(d))
    v = 0.0
    for j in range(d):
        v += max(0.0, (q[j] - inst.existencias[j]) / inst.existencias[j])
        v += max(0.0, -q[j] / inst.demanda_actual[j])
    for f in range(len(inst.familias)):
        miembros = sorted((j for j in range(d) if inst.familia[j] == f), key=lambda j: inst.nivel[j])
        for i, s in zip(miembros[:-1], miembros[1:]):
            v += max(0.0, ((1 + inst.brecha_escalera) * p[i] - p[s]) / inst.precio_actual[s])
    indice = sum(inst.pesos_indice[j] * p[j] / inst.precio_actual[j] for j in range(d))
    v += max(0.0, indice - (1 + inst.holgura_indice))
    en_caja = all(inst.piso[j] <= p[j] <= inst.techo[j] for j in range(d))
    return {"utilidad": valor, "violacion": v, "demanda": q, "indice": indice, "en_caja": en_caja}

## C. Óptimo exacto y líneas base

### C.1 Óptimo global certificado

Como la utilidad es estrictamente cóncava y el conjunto factible es un poliedro convexo, el problema tiene **un único óptimo global**, y cualquier punto que cumpla las condiciones de Karush-Kuhn-Tucker (KKT) lo es (Boyd y Vandenberghe, 2004). La referencia se obtiene en dos pasos:

1. **SLSQP** desde cuatro arranques —precio actual, piso, techo y punto medio— con tolerancia de $10^{-15}$. Es un método de programación cuadrática secuencial, rápido, pero sus puntos llegan a violar alguna regla hasta en $10^{-6}$.
2. **Certificado por conjunto activo.** Con las restricciones activas en el mejor arranque se resuelve el sistema lineal KKT exacto,
$$\begin{pmatrix}2B & A_{\mathcal A}^{\top}\\ A_{\mathcal A} & 0\end{pmatrix}\begin{pmatrix}p\\ \lambda\end{pmatrix}=\begin{pmatrix}a+Bc\\ b_{\mathcal A}\end{pmatrix},$$
y se ajusta el conjunto activo hasta que los multiplicadores sean no negativos y ninguna restricción quede violada. El resultado cumple KKT a precisión de máquina.

Los multiplicadores tienen lectura comercial: $\lambda_i$ es cuánta utilidad se ganaría por relajar la regla $i$ en una unidad de su escala. El algoritmo genético **nunca** usa esta referencia: no lo inicializa, no lo guía y no lo detiene.

In [6]:
def gradiente_utilidad(inst: Instancia, p: np.ndarray) -> np.ndarray:
    return inst.a + inst.B @ inst.costo - 2.0 * inst.B @ p


def matriz_completa(inst: Instancia):
    """Todas las restricciones como A p <= b: reglas comerciales, techos y pisos."""
    d = inst.d
    A = np.vstack([inst.G, np.eye(d), -np.eye(d)])
    b = np.concatenate([inst.h, inst.techo, -inst.piso])
    nombres = list(inst.tipo) + [("techo", j) for j in range(d)] + [("piso", j) for j in range(d)]
    return A, b, nombres


def certificar_kkt(inst: Instancia, x: np.ndarray, tolerancia: float = 1e-9, max_iter: int = 100) -> dict:
    """Resuelve el sistema KKT sobre un conjunto activo y lo corrige hasta cumplir todas las condiciones."""
    d = inst.d
    A, b, nombres = matriz_completa(inst)
    H, g = 2.0 * inst.B, inst.a + inst.B @ inst.costo
    activos = set(np.flatnonzero(b - A @ x < 1e-7).tolist())
    for iteracion in range(1, max_iter + 1):
        act = np.array(sorted(activos), dtype=int)
        m = len(act)
        K = np.block([[H, A[act].T], [A[act], np.zeros((m, m))]])
        solucion = np.linalg.lstsq(K, np.concatenate([g, b[act]]), rcond=None)[0]
        x, lam = solucion[:d], solucion[d:]
        holgura = b - A @ x
        escala = max(1.0, float(np.abs(lam).max())) if m else 1.0
        if m and lam.min() < -tolerancia * escala:
            activos.discard(int(act[np.argmin(lam)]))
            continue
        if holgura.min() < -tolerancia:
            activos.add(int(np.argmin(holgura)))
            continue
        residuo = float(np.max(np.abs(g - H @ x - A[act].T @ lam))) if m else float(np.max(np.abs(g - H @ x)))
        return {"precios": x, "utilidad": float(utilidad(inst, x)[0]), "activas": [nombres[i] for i in act],
                "multiplicadores": lam, "residuo": residuo, "max_violacion": float(max(0.0, -holgura.min())),
                "iteraciones": iteracion}
    raise RuntimeError(f"{inst.nombre}: el conjunto activo no convergió; no hay certificado.")


def resolver_exacto(inst: Instancia) -> dict:
    """Óptimo global de la utilidad: SLSQP desde cuatro arranques y certificado KKT."""
    objetivo = lambda p: -utilidad(inst, p)[0]
    gradiente = lambda p: -gradiente_utilidad(inst, p)
    reglas = [{"type": "ineq", "fun": lambda p: inst.h - inst.G @ p, "jac": lambda p: -inst.G}]
    cotas = list(zip(inst.piso, inst.techo))
    arranques = {"precio actual": inst.precio_actual, "piso": inst.piso, "techo": inst.techo,
                 "punto medio": (inst.piso + inst.techo) / 2}
    inicio = time.perf_counter()
    puntos = {}
    for nombre, x0 in arranques.items():
        r = minimize(objetivo, x0, jac=gradiente, method="SLSQP", bounds=cotas, constraints=reglas,
                     options={"ftol": 1e-15, "maxiter": 5000})
        puntos[nombre] = np.clip(r.x, inst.piso, inst.techo)
    mejor = max(puntos.values(), key=lambda x: utilidad(inst, x)[0] - 1e9 * violacion(inst, x)[0])
    cert = certificar_kkt(inst, mejor)
    cert["segundos"] = time.perf_counter() - inicio
    cert["slsqp"] = {k: (float(utilidad(inst, x)[0]), float(violacion(inst, x)[0])) for k, x in puntos.items()}
    if cert["max_violacion"] > 1e-9 or cert["residuo"] > 1e-6 * max(1.0, np.abs(inst.a).max()):
        raise RuntimeError(f"{inst.nombre}: el certificado no alcanzó la tolerancia.")
    return cert

### C.2 Líneas base

Tres referencias que no son algoritmos genéticos, para saber si el algoritmo aporta algo:

- **Precios actuales.** La regla vigente de costo más 45 %. Es factible por construcción y es lo que la empresa obtendría sin hacer nada.
- **Ajuste parejo.** Subir o bajar **todos** los precios el mismo porcentaje $t$, recortado al piso y al techo de cada producto. Es la decisión que se toma con una hoja de cálculo. Se busca el mejor $t\in[-25\,\%,+20\,\%]$ en una rejilla de 0.05 puntos que incluye $t=0$.
- **Búsqueda aleatoria.** Muestreo uniforme independiente en la caja de precios (el muestreo de M01), con **exactamente el mismo número de evaluaciones** que el algoritmo genético y la misma regla M08 para quedarse con la mejor. Mide cuánto del resultado se obtendría sin evolución.

In [7]:
def ajuste_parejo(inst: Instancia, paso: float = 0.0005) -> dict:
    """Mejor cambio porcentual común a todos los precios, recortado a la caja de cada producto."""
    t = np.round(np.arange(-0.25, ALZA_MAXIMA + paso / 2, paso), 4)
    X = np.clip(inst.precio_actual[None, :] * (1 + t[:, None]), inst.piso, inst.techo)
    U = np.where(violacion(inst, X) <= TOL_FACTIBLE, utilidad(inst, X), -np.inf)
    i = int(np.argmax(U))
    return {"t": float(t[i]), "precios": X[i], "utilidad": float(U[i])}


def busqueda_aleatoria(inst: Instancia, semilla: int, evaluaciones: int, N: int = 100, por_bloque: int = 98,
                       z_ref: float = None) -> dict:
    """Muestreo uniforme en la caja con presupuesto idéntico; registra con la granularidad del AG."""
    rng = np.random.default_rng(semilla)
    mejor_V, mejor_U, mejor_x = np.inf, -np.inf, None
    usadas, trayectoria = 0, []
    bloque = N
    while usadas + bloque <= evaluaciones:
        X = inst.piso + rng.random((bloque, inst.d)) * (inst.techo - inst.piso)
        ev = etapa_2_evaluar_M08(inst, X)
        V, U = ev["V"], ev["utilidad"]
        usadas += bloque
        i = int(orden_M08(ev)[0])
        if (V[i], -U[i]) < (mejor_V, -mejor_U):
            mejor_V, mejor_U, mejor_x = float(V[i]), float(U[i]), X[i].copy()
        trayectoria.append({"evaluaciones": usadas, "mejor_factible": mejor_U if mejor_V <= 0 else np.nan})
        bloque = por_bloque
    factible = mejor_V <= 0
    return {"utilidad": mejor_U if factible else np.nan, "factible": factible, "precios": mejor_x,
            "violacion_minima": mejor_V, "evaluaciones": usadas, "trayectoria": trayectoria}

## D. Las siete etapas

La guía de los treinta métodos vista en clase reparte las alternativas en siete etapas; en cada una se elige un método. Aquí se fijan cinco y **se realizan de tres formas** las etapas 4 y 5. Al igual que en la Actividad 03, cada función devuelve datos nuevos y no modifica a sus padres, todo el azar de una corrida sale de **un solo generador** sembrado una vez, y el orden efectivo es: 1 → 2 → control 7 → 3 → 4 → 5 → repetir 2 sobre los hijos → 6 → control 7.

**Por qué estas dos etapas.** Con genes reales, la recombinación y la mutación son las únicas etapas en que la guía ofrece tres métodos nativos de la representación y en que las tres formas son mecanismos realmente distintos de generar precios nuevos. En las otras cinco, la representación, las restricciones o el propio diseño experimental dejan una sola opción razonable (sección D.1, D.2, D.3, D.6 y D.7). Además, recombinación y mutación **se esperan interdependientes**: M14 no crea valores nuevos y depende de la mutación para afinar, mientras que M15 y M16 sí los crean. Por eso se cruzan en un diseño factorial y no se comparan una a la vez.

### D.1 Etapa 1 — Inicialización: M03, hipercubo latino (fija)

| Alternativa | ¿Aplica a precios reales en una caja? | Decisión |
|---|---|---|
| M01 uniforme en caja | Sí | Descartada: con $N=100$ deja huecos al azar en el rango de algunos precios |
| M02 Bernoulli | No: produce bits | Descartada |
| **M03 hipercubo latino** | **Sí** | **Elegida** |
| M04 Dirichlet | No: impone que los genes sumen uno | Descartada |

Cada precio se divide en $N$ estratos de igual ancho y cada estrato recibe exactamente un individuo:
$$x_{ij}=\ell_j+\frac{\pi_j(i)+U_{ij}}{N}\,(u_j-\ell_j),\qquad \pi_j\ \text{permutación aleatoria de}\ \{0,\dots,N-1\},\quad U_{ij}\sim\mathcal U(0,1).$$
A diferencia del agente viajero de la Actividad 02, aquí el gen se usa tal cual, sin decodificador, de modo que estratificar sí garantiza que la población inicial recorra el rango completo de **cada** precio (McKay et al., 1979). Como la población inicial sale del generador antes que cualquier otro sorteo, **las nueve configuraciones parten de la misma población con la misma semilla**: es lo que permite emparejar las comparaciones.

In [8]:
def etapa_1_inicializar_M03(N: int, inst: Instancia, rng: np.random.Generator) -> np.ndarray:
    """M03: hipercubo latino aleatorizado en la caja [piso, techo]."""
    d = inst.d
    estratos = rng.permuted(np.tile(np.arange(N), (d, 1)), axis=1).T     # cada columna, una permutación
    U = (estratos + rng.random((N, d))) / N
    return inst.piso + U * (inst.techo - inst.piso)

### D.2 Etapa 2 — Evaluación: M08, prioridad de factibilidad (fija)

| Alternativa | Decisión |
|---|---|
| M05 objetivo directo | Descartada: ignora existencias, escalera e índice, y entregaría precios imposibles de aplicar |
| M06 suma ponderada | Descartada: el problema tiene un solo objetivo |
| M07 penalización | Descartada: exige un coeficiente; en la Actividad 03, con un coeficiente bajo, las 30 corridas entregaron soluciones inviables |
| **M08 prioridad de factibilidad** | **Elegida** |

A cada vector de precios se le asigna la **clave** $(V(p),\,-\pi(p))$ y se compara en orden lexicográfico (Deb, 2000): cualquier factible supera a cualquier inviable; entre inviables gana el de menor violación, y entre factibles el de mayor utilidad. Se mantiene fija porque **las seis formas comparadas tienen que competir con la misma vara**. Con M08, además, el mejor individuo es siempre la mejor solución factible en cuanto existe alguna.

La factibilidad se decide con una tolerancia de $10^{-12}$ en las unidades relativas de $V$: una violación menor se trata como cero. Esa tolerancia sólo absorbe el redondeo de punto flotante, porque el mismo vector evaluado en un lote o por separado puede dar $0$ o $10^{-17}$. Equivale a exceder unas existencias de mil unidades en una milmillonésima de unidad.

In [9]:
def etapa_2_evaluar_M08(inst: Instancia, X: np.ndarray) -> dict:
    """M08: violación y utilidad; la comparación usa la clave lexicográfica (V, -utilidad)."""
    V, U = violacion(inst, X), utilidad(inst, X)
    V = np.where(V <= TOL_FACTIBLE, 0.0, V)
    return {"V": V, "utilidad": U, "factible": V == 0.0}


def orden_M08(ev: dict) -> np.ndarray:
    """Índices del mejor al peor: menor violación y, a igual violación, mayor utilidad."""
    return np.lexsort((-ev["utilidad"], ev["V"]))


def rangos_densos_M08(ev: dict) -> np.ndarray:
    """Rango 0 = mejor; claves idénticas comparten rango."""
    o = orden_M08(ev)
    V, U = ev["V"][o], ev["utilidad"][o]
    nuevo = np.ones(len(o), dtype=bool)
    nuevo[1:] = (V[1:] != V[:-1]) | (U[1:] != U[:-1])
    rangos = np.empty(len(o), dtype=int)
    rangos[o] = np.cumsum(nuevo) - 1
    return rangos

### D.3 Etapa 3 — Selección: M12, torneo con reemplazo (fija)

La ruleta (M09) y el muestreo universal estocástico (M10) exigen aptitudes positivas y proporcionales, que la clave lexicográfica de M08 no tiene. La selección por rangos (M11) sería compatible, pero el torneo hace lo mismo con una comparación directa y un solo parámetro (Blickle y Thiele, 1996). Se usa $k=2$, el torneo binario, el mismo valor que la calibración de la Actividad 03 eligió. **No se calibra aquí**: es común a las nueve configuraciones, y el presupuesto de calibración se reserva para los parámetros propios de los operadores comparados. Los empates se resuelven por la primera aparición sorteada.

In [10]:
def torneo_desde_participantes(rangos: np.ndarray, participantes: np.ndarray) -> np.ndarray:
    """Gana el de menor rango; en empate, el primero sorteado (argmin devuelve la primera aparición)."""
    return participantes[np.arange(len(participantes)), np.argmin(rangos[participantes], axis=1)]


def etapa_3_seleccionar_M12(rangos: np.ndarray, n: int, k: int, rng: np.random.Generator) -> np.ndarray:
    """M12: n torneos de tamaño k con reemplazo."""
    return torneo_desde_participantes(rangos, rng.integers(0, len(rangos), size=(n, k)))

### D.4 Etapa 4 — Recombinación: las tres formas comparadas

La guía ofrece cuatro métodos. Los tres elegidos representan **tres mecanismos distintos de combinar dos precios**: tomar el de uno u otro padre, promediarlos o muestrear alrededor de ambos.

| Método | Mecanismo | ¿Crea valores nuevos? | Parámetro propio |
|---|---|---|---|
| **M14 uniforme por máscara** | Intercambio: cada gen se hereda de uno u otro padre | No | — ($q=0.5$ por simetría) |
| **M15 aritmética** | Interpolación: el hijo cae en el segmento entre los padres | Sí, siempre entre ellos | — ($\alpha\sim\mathcal U(0,1)$ por pareja) |
| **M16 BLX-α** | Extrapolación: el hijo se muestrea en un intervalo ampliado | Sí, también fuera de ellos | $\alpha$ (calibrado) |
| M13 un punto | Intercambio por bloques contiguos | No | Descartada: es el mismo mecanismo que M14, pero sus bloques dependen de cómo se numeraron los productos |

$$\textbf{M14:}\quad h^{(1)}_j=m_ja_j+(1-m_j)b_j,\quad h^{(2)}_j=(1-m_j)a_j+m_jb_j,\qquad m_j\sim\operatorname{Bernoulli}(0.5)$$

$$\textbf{M15:}\quad h^{(1)}=\alpha a+(1-\alpha)b,\quad h^{(2)}=(1-\alpha)a+\alpha b,\qquad \alpha\sim\mathcal U(0,1)\ \text{por pareja}$$

$$\textbf{M16:}\quad h_j\sim\mathcal U\big[\min(a_j,b_j)-\alpha I_j,\ \max(a_j,b_j)+\alpha I_j\big],\quad I_j=|a_j-b_j|,\ \text{recortado a}\ [\ell_j,u_j]$$

Tres propiedades teóricas orientan las hipótesis de la sección G:

- **M14 y M15 conservan la suma de los padres**, $h^{(1)}+h^{(2)}=a+b$; M16 no.
- **M15 nunca sale del conjunto factible si los dos padres son factibles.** El conjunto factible es convexo y el hijo es una combinación convexa de los padres. Con restricciones lineales como éstas, es la única de las tres formas que respeta la factibilidad por construcción.
- **M15 contrae la población.** Con $\alpha\sim\mathcal U(0,1)$, la distancia entre los hijos es $|2\alpha-1|$ veces la de los padres, en promedio la mitad; la varianza se pierde generación tras generación, como ya se observó en la Actividad 02. BLX-α (Eshelman y Schaffer, 1993) amplía el intervalo justamente para compensar esa pérdida.

La pareja se cruza con probabilidad $p_c=0.9$; si no, los hijos son copias de los padres.

In [11]:
def cruce_con_mascara(A: np.ndarray, B: np.ndarray, mascara: np.ndarray):
    """Hijo 1 toma de A donde la máscara vale 1 y de B donde vale 0; el hijo 2, al revés."""
    m = mascara.astype(bool)
    return np.where(m, A, B), np.where(m, B, A)


def etapa_4_recombinar_M14(A: np.ndarray, B: np.ndarray, rng: np.random.Generator, q: float = 0.5):
    """M14: cruce uniforme por máscara; cada gen se hereda de uno u otro padre."""
    return cruce_con_mascara(A, B, rng.random(A.shape) < q)


def combinacion_aritmetica(A: np.ndarray, B: np.ndarray, alfa: np.ndarray):
    """Hijos alfa·A + (1 - alfa)·B y (1 - alfa)·A + alfa·B; alfa es un escalar o una columna."""
    return alfa * A + (1 - alfa) * B, (1 - alfa) * A + alfa * B


def etapa_4_recombinar_M15(A: np.ndarray, B: np.ndarray, rng: np.random.Generator):
    """M15: recombinación aritmética completa, con un alfa uniforme por pareja."""
    return combinacion_aritmetica(A, B, rng.random((A.shape[0], 1)))


def etapa_4_recombinar_M16(A: np.ndarray, B: np.ndarray, rng: np.random.Generator, alfa: float,
                           piso: np.ndarray, techo: np.ndarray):
    """M16: BLX-alfa; muestreo uniforme en el intervalo de los padres ampliado alfa·I por cada lado."""
    bajo, alto = np.minimum(A, B), np.maximum(A, B)
    I = alto - bajo
    H1 = rng.uniform(bajo - alfa * I, alto + alfa * I)
    H2 = rng.uniform(bajo - alfa * I, alto + alfa * I)
    return np.clip(H1, piso, techo), np.clip(H2, piso, techo)


RECOMBINAR = {
    "M14": lambda A, B, rng, inst, par: etapa_4_recombinar_M14(A, B, rng),
    "M15": lambda A, B, rng, inst, par: etapa_4_recombinar_M15(A, B, rng),
    "M16": lambda A, B, rng, inst, par: etapa_4_recombinar_M16(A, B, rng, par["alfa"], inst.piso, inst.techo),
}

### D.5 Etapa 5 — Mutación: las tres formas comparadas

M17, la inversión de bits, no aplica a precios reales. Los otros tres métodos de la guía son tres mecanismos distintos: **reiniciar** el gen en todo su intervalo, **perturbarlo** con un ruido sin cota y recortar, o **perturbarlo con una distribución que ya respeta las cotas**.

| Método | Mecanismo | Alcance | Qué pasa en la cota | Parámetro propio |
|---|---|---|---|---|
| **M18 reinicio uniforme** | Sortea un precio nuevo en $[\ell_j,u_j]$ | Global | Nada: nunca sale | — |
| **M19 gaussiana acotada** | $x_j+\sigma(u_j-\ell_j)\,z$, $z\sim\mathcal N(0,1)$, recortada | Local | Acumula masa exactamente en la cota | $\sigma$ (calibrado) |
| **M20 polinomial acotada** | Perturbación polinomial cuyo alcance se ajusta a la distancia a cada cota | Local | Nunca sale, sin recorte | $\eta_m$ (calibrado) |

La mutación polinomial acotada (Deb y Goyal, 1996; Deb y Deb, 2014) usa $\delta_1=(x-\ell)/(u-\ell)$, $\delta_2=(u-x)/(u-\ell)$ y $r\sim\mathcal U(0,1)$:
$$\delta_q=\begin{cases}\big[2r+(1-2r)(1-\delta_1)^{\eta_m+1}\big]^{1/(\eta_m+1)}-1, & r\le 0.5\\[2pt] 1-\big[2(1-r)+2(r-0.5)(1-\delta_2)^{\eta_m+1}\big]^{1/(\eta_m+1)}, & r>0.5\end{cases}\qquad y=x+\delta_q\,(u-\ell).$$
Con $r\to0$ el gen llega exactamente al piso y con $r\to1$ al techo; nunca los rebasa. Un $\eta_m$ grande concentra la perturbación cerca de $x$.

En las tres, la probabilidad de mutar es **por gen**, común a las nueve configuraciones: $p_m=1/d$, un precio en promedio por hijo.

In [12]:
def etapa_5_mutar_M18(H: np.ndarray, pm: float, rng: np.random.Generator, piso, techo) -> np.ndarray:
    """M18: con probabilidad pm por gen, sortea un precio nuevo uniforme en [piso, techo]."""
    m = rng.random(H.shape) < pm
    return np.where(m, piso + rng.random(H.shape) * (techo - piso), H)


def etapa_5_mutar_M19(H: np.ndarray, pm: float, rng: np.random.Generator, piso, techo, sigma: float) -> np.ndarray:
    """M19: perturbación gaussiana con desviación sigma·(techo - piso), recortada a la caja."""
    m = rng.random(H.shape) < pm
    Y = np.clip(H + sigma * (techo - piso) * rng.standard_normal(H.shape), piso, techo)
    return np.where(m, Y, H)


def etapa_5_mutar_M20(H: np.ndarray, pm: float, rng: np.random.Generator, piso, techo, eta: float) -> np.ndarray:
    """M20: mutación polinomial acotada; el resultado queda en [piso, techo] por construcción."""
    m = rng.random(H.shape) < pm
    R = techo - piso
    d1, d2 = (H - piso) / R, (techo - H) / R
    r = rng.random(H.shape)
    e1 = eta + 1.0
    izquierda = (2 * r + (1 - 2 * r) * (1 - d1) ** e1) ** (1 / e1) - 1
    derecha = 1 - (2 * (1 - r) + 2 * (r - 0.5) * (1 - d2) ** e1) ** (1 / e1)
    Y = H + np.where(r <= 0.5, izquierda, derecha) * R
    Y = np.minimum(np.maximum(Y, piso), techo)     # sólo absorbe el error de redondeo de punto flotante
    return np.where(m, Y, H)


MUTAR = {
    "M18": lambda H, pm, rng, inst, par: etapa_5_mutar_M18(H, pm, rng, inst.piso, inst.techo),
    "M19": lambda H, pm, rng, inst, par: etapa_5_mutar_M19(H, pm, rng, inst.piso, inst.techo, par["sigma"]),
    "M20": lambda H, pm, rng, inst, par: etapa_5_mutar_M20(H, pm, rng, inst.piso, inst.techo, par["eta"]),
}
PARAMETRO_DE = {"M14": None, "M15": None, "M16": "alfa", "M18": None, "M19": "sigma", "M20": "eta"}

### D.6 Etapa 6 — Reemplazo: M22, generacional con elitismo (fija)

$$P^{t+1}=\operatorname{Élite}_e(P^t)\,\Vert\,H^t,\qquad |H^t|=N-e,\qquad e=2.$$

El reemplazo generacional puro (M21) puede perder la mejor solución; $(\mu,\lambda)$ (M23) también, porque descarta a los padres. El estacionario competitivo (M24) cambia cuántas evaluaciones hay por transición. Con dos élites y la clave de M08, **la mejor clave nunca empeora** y la mejor solución factible encontrada no se pierde. Es la misma elección de la Actividad 03.

In [13]:
def etapa_6_reemplazar_M22(P: np.ndarray, ev: dict, H: np.ndarray, evH: dict, e: int):
    """M22: conserva las e mejores claves de la población actual y completa con los hijos."""
    elite = orden_M08(ev)[:e]
    P_nueva = np.vstack([P[elite], H])
    ev_nueva = {k: np.concatenate([ev[k][elite], evH[k]]) for k in ev}
    return P_nueva, ev_nueva

### D.7 Etapa 7 — Criterio de paro: M26, presupuesto de evaluaciones (fija)

Aquí el criterio de paro lo decide **el diseño experimental**, no el problema. En la Actividad 03 se usó el estancamiento (M28), que detiene cada corrida en un momento distinto. Para comparar operadores eso es un defecto: una configuración que se estanca pronto gastaría menos evaluaciones y su menor calidad no podría separarse de su menor esfuerzo. Con M26 **las nueve configuraciones gastan exactamente el mismo número de evaluaciones**, y cualquier diferencia de calidad se atribuye a los operadores.

$$\text{parar en } t\iff E_t+(N-e)>E_{\max},\qquad E_t=N+t\,(N-e).$$

No se inicia una generación que no quepa completa. Con $E_{\max}=20\,000$, $N=100$ y $e=2$, cada corrida hace 203 generaciones y 19 994 evaluaciones. El costo de esa decisión también se mide: la trayectoria completa de cada corrida permite leer la calidad a la cuarta parte y a la mitad del presupuesto.

In [14]:
def etapa_7_parar_M26(evaluaciones: int, por_generacion: int, E_max: int) -> bool:
    """M26: se detiene si la siguiente generación completa no cabe en el presupuesto."""
    return evaluaciones + por_generacion > E_max

### D.8 Integración: el algoritmo completo

La función integradora recibe la recombinación y la mutación por nombre, junto con sus parámetros, y registra en cada generación la mejor utilidad factible, la fracción factible de la población y de los hijos, la **dispersión** de la población y la fracción de genes que están exactamente en una cota. La dispersión es la desviación estándar media de los precios, normalizada por el ancho de cada caja:
$$D(P)=\frac1d\sum_j\frac{\operatorname{sd}(x_{\cdot j})}{u_j-\ell_j}.$$
Al terminar comprueba tres contratos y lanza un error si alguno falla: que el contador de evaluaciones sea $N+G(N-e)$, que la mejor clave no haya empeorado nunca, y que la solución entregada, recalculada con la evaluación escalar, sea factible, esté en la caja y tenga la utilidad informada.

In [15]:
def dispersion(P: np.ndarray, inst: Instancia) -> float:
    return float(np.mean(P.std(axis=0) / (inst.techo - inst.piso)))


def algoritmo_genetico(inst: Instancia, cruce: str, mutacion: str, semilla: int, parametros: dict = None,
                       N: int = 100, k: int = 2, e: int = 2, pc: float = 0.9, pm: float = None,
                       E_max: int = 20_000, historial: bool = True) -> dict:
    """Siete etapas: M03, M08, M12, recombinación y mutación elegidas, M22 y M26."""
    par = dict(parametros or {})
    d = inst.d
    pm = 1.0 / d if pm is None else pm
    rng = np.random.default_rng(semilla)                  # un solo generador por corrida
    inicio = time.perf_counter()
    P = etapa_1_inicializar_M03(N, inst, rng)             # ETAPA 1: M03
    ev = etapa_2_evaluar_M08(inst, P)                     # ETAPA 2: M08
    evaluaciones, generacion = N, 0
    hijos, parejas = N - e, (N - e + 1) // 2
    mejor = {"utilidad": -np.inf, "precios": None, "evaluaciones": None}
    registro, clave_anterior = [], (np.inf, np.inf)

    def registrar(frac_hijos: float):
        nonlocal clave_anterior
        i = int(orden_M08(ev)[0])
        clave = (float(ev["V"][i]), float(-ev["utilidad"][i]))
        if clave > clave_anterior:
            raise RuntimeError("La mejor clave empeoró: el elitismo no se cumplió.")
        clave_anterior = clave
        if ev["factible"][i] and ev["utilidad"][i] > mejor["utilidad"]:
            mejor.update(utilidad=float(ev["utilidad"][i]), precios=P[i].copy(), evaluaciones=evaluaciones)
        if historial:
            en_cota = np.mean((P == inst.piso) | (P == inst.techo))
            registro.append({"generacion": generacion, "evaluaciones": evaluaciones,
                             "mejor_factible": mejor["utilidad"] if mejor["precios"] is not None else np.nan,
                             "fraccion_factible": float(ev["factible"].mean()), "fraccion_hijos_factibles": frac_hijos,
                             "dispersion": dispersion(P, inst), "en_cota": float(en_cota)})

    registrar(np.nan)
    while not etapa_7_parar_M26(evaluaciones, hijos, E_max):                   # ETAPA 7: M26
        rangos = rangos_densos_M08(ev)
        indices = etapa_3_seleccionar_M12(rangos, 2 * parejas, k, rng)         # ETAPA 3: M12
        A, B = P[indices[0::2]], P[indices[1::2]]
        H1, H2 = RECOMBINAR[cruce](A, B, rng, inst, par)                       # ETAPA 4: M14, M15 o M16
        cruza = rng.random(parejas) < pc
        H1 = np.where(cruza[:, None], H1, A)
        H2 = np.where(cruza[:, None], H2, B)
        H = np.empty((2 * parejas, d))
        H[0::2], H[1::2] = H1, H2
        H = MUTAR[mutacion](H[:hijos], pm, rng, inst, par)                     # ETAPA 5: M18, M19 o M20
        evH = etapa_2_evaluar_M08(inst, H)                                     # repetir ETAPA 2
        evaluaciones += hijos
        P, ev = etapa_6_reemplazar_M22(P, ev, H, evH, e)                       # ETAPA 6: M22
        generacion += 1
        registrar(float(evH["factible"].mean()))

    if evaluaciones != N + generacion * (N - e) or evaluaciones > E_max:
        raise RuntimeError("El contador de evaluaciones no coincide con N + G(N - e).")
    if mejor["precios"] is not None:
        auditoria = evaluar_escalar(inst, mejor["precios"])
        if (auditoria["violacion"] > 10 * TOL_FACTIBLE or not auditoria["en_caja"]
                or abs(auditoria["utilidad"] - mejor["utilidad"]) > 1e-8 * abs(mejor["utilidad"])):
            raise RuntimeError("La solución entregada no coincide con su evaluación escalar.")
    return {"utilidad": mejor["utilidad"] if mejor["precios"] is not None else np.nan,
            "precios": mejor["precios"], "eval_mejor": mejor["evaluaciones"], "evaluaciones": evaluaciones,
            "generaciones": generacion, "segundos": time.perf_counter() - inicio, "historial": registro,
            "poblacion": P, "fraccion_factible": float(ev["factible"].mean()), "dispersion": dispersion(P, inst),
            "en_cota": float(np.mean((P == inst.piso) | (P == inst.techo))),
            "parametros": {"cruce": cruce, "mutacion": mutacion, **par, "N": N, "k": k, "e": e, "pc": pc,
                           "pm": pm, "E_max": E_max, "semilla": semilla}}

## E. Pruebas automatizadas

Dieciséis pruebas se ejecutan **antes** de cualquier experimento; si una falla, el cuadernillo se detiene. Dos de ellas contrastan la referencia exacta con problemas resueltos a mano, de modo que el certificado KKT se comprueba contra respuestas conocidas antes de usarlo como vara. Las pruebas de los operadores verifican las propiedades que la sección D.4 y D.5 invocan para justificarlos.

| Prueba | Qué comprueba |
|---|---|
| 01 | La validación rechaza una matriz $B$ no definida positiva y un piso por debajo del costo |
| 02 | Evaluación vectorizada idéntica a la escalar en 500 vectores de precios |
| 03 | Un producto, $q=100-2p$, $c=10$: óptimo $p^\ast=30$ y $\pi^\ast=800$; con techo 25, $\pi^\ast=750$ y multiplicador 20 |
| 04 | Dos sustitutos simétricos, $q_1=100-2p_1+p_2$: óptimo $p^\ast=55$ y $\pi^\ast=4\,050$ |
| 05 | Certificado KKT en una instancia generada: factibilidad, multiplicadores no negativos; ni los arranques ni 20 000 vecinos factibles del óptimo lo superan |
| 06 | M03: un individuo por estrato en cada precio, dentro de la caja, reproducible |
| 07 | M08: el factible gana al inviable aunque valga menos; la penalización con coeficiente bajo elige al inviable |
| 08 | M12: el torneo $\{C,A,C\}$ lo gana $A$; un empate lo gana el primero sorteado |
| 09 | M14: la máscara $(1,0,1,0)$ sobre $(1,2,3,4)$ y $(5,6,7,8)$ da $(1,6,3,8)$ y $(5,2,7,4)$; se conserva la suma |
| 10 | M15: hijos en el segmento entre los padres, suma conservada y, de padres factibles, **hijos factibles** |
| 11 | M16: con $\alpha=0$ los hijos quedan entre los padres; con $\alpha>0$ en el intervalo ampliado; el recorte deja masa en la cota |
| 12 | M18: $p_m=0$ no cambia nada; $p_m=1$ reinicia todo dentro de la caja con media en el centro |
| 13 | M19: siempre en la caja; desviación $\sigma(u-\ell)$; desde el techo, la mitad de las mutaciones queda exactamente en el techo |
| 14 | M20: nunca sale de la caja en 200 000 sorteos desde el piso, el techo y el centro; un $\eta_m$ mayor perturba menos |
| 15 | Integración: presupuesto $N+G(N-e)$, misma población inicial para las nueve configuraciones, reproducibilidad, elitismo y auditoría |
| 16 | Líneas base: búsqueda aleatoria con presupuesto idéntico; ajuste parejo factible y no peor que los precios actuales |

In [16]:
def instancia_manual(costo, precio, q0, B, existencias, piso, techo, holgura=10.0) -> Instancia:
    """Instancia pequeña para pruebas: cada producto en su propia familia (sin escalera)."""
    d = len(costo)
    inst = ensamblar_instancia("manual", tuple(f"F{f}" for f in range(d)), np.arange(d), np.zeros(d, int),
                               costo, precio, q0, B, existencias, piso, techo, BRECHA_ESCALERA, holgura)
    validar_instancia(inst)
    return inst


INST_PRUEBA = generar_instancia("T06-D", 2, "D", 7)
INST_PRUEBA_F = generar_instancia("T15-F", 5, "F", 11)
RESULTADOS_PRUEBAS = []


def prueba(numero: str, descripcion: str):
    def decorador(funcion):
        funcion()
        RESULTADOS_PRUEBAS.append({"prueba": numero, "descripcion": descripcion, "resultado": "pasó"})
        return funcion
    return decorador


def muestras_cercanas(inst: Instancia, centro: np.ndarray, n: int, semilla: int, escala: float = 0.08) -> np.ndarray:
    """Vectores de precios alrededor de un centro, recortados a la caja; mezclan factibles e inviables."""
    rng = np.random.default_rng(semilla)
    return np.clip(centro * (1 + escala * rng.uniform(-1, 1, (n, inst.d))), inst.piso, inst.techo)


def muestras_factibles(inst: Instancia, n: int, semilla: int) -> np.ndarray:
    X = muestras_cercanas(inst, inst.precio_actual, 50_000, semilla)
    X = X[violacion(inst, X) <= TOL_FACTIBLE]
    assert len(X) >= n, "no hay suficientes muestras factibles para la prueba"
    return X[:n]


@prueba("01", "Validación de instancias")
def _():
    for malo in (replace(INST_PRUEBA, B=-INST_PRUEBA.B), replace(INST_PRUEBA, piso=INST_PRUEBA.costo * 0.9)):
        try:
            validar_instancia(malo)
        except ValueError:
            continue
        raise AssertionError("la validación aceptó una instancia inválida")


@prueba("02", "Evaluación vectorizada idéntica a la escalar")
def _():
    inst = INST_PRUEBA_F
    X = np.vstack([inst.piso + np.random.default_rng(1).random((250, inst.d)) * (inst.techo - inst.piso),
                   muestras_cercanas(inst, inst.precio_actual, 250, 1)])
    U, V = utilidad(inst, X), violacion(inst, X)
    for i in range(len(X)):
        r = evaluar_escalar(inst, X[i])
        assert abs(r["utilidad"] - U[i]) <= 1e-9 * abs(U[i]) and abs(r["violacion"] - V[i]) <= 1e-12
    assert 0 < np.mean(V <= TOL_FACTIBLE) < 1, "la muestra debe mezclar factibles e inviables"


@prueba("03", "Un producto resuelto a mano")
def _():
    base = dict(costo=[10.0], precio=[20.0], q0=[60.0], B=[[2.0]], existencias=[1000.0], piso=[11.0])
    r = resolver_exacto(instancia_manual(**base, techo=[60.0]))
    assert abs(r["precios"][0] - 30) < 1e-9 and abs(r["utilidad"] - 800) < 1e-7 and not r["activas"]
    r = resolver_exacto(instancia_manual(**base, techo=[25.0]))
    assert abs(r["precios"][0] - 25) < 1e-9 and abs(r["utilidad"] - 750) < 1e-7
    assert r["activas"] == [("techo", 0)] and abs(r["multiplicadores"][0] - 20) < 1e-7


@prueba("04", "Dos sustitutos simétricos resueltos a mano")
def _():
    inst = instancia_manual(costo=[10.0, 10.0], precio=[20.0, 20.0], q0=[80.0, 80.0], B=[[2.0, -1.0], [-1.0, 2.0]],
                            existencias=[1000.0, 1000.0], piso=[11.0, 11.0], techo=[100.0, 100.0])
    r = resolver_exacto(inst)
    assert np.allclose(r["precios"], 55, atol=1e-9) and abs(r["utilidad"] - 4050) < 1e-6


@prueba("05", "Certificado KKT en una instancia generada")
def _():
    inst = INST_PRUEBA_F
    r = resolver_exacto(inst)
    assert r["max_violacion"] <= 1e-9 and r["residuo"] <= 1e-6 * np.abs(inst.a).max()
    assert len(r["multiplicadores"]) == 0 or r["multiplicadores"].min() >= -1e-9 * max(1, np.abs(r["multiplicadores"]).max())
    for u, v in r["slsqp"].values():
        assert v > 1e-9 or u <= r["utilidad"] * (1 + 1e-9), "un arranque factible superó al certificado"
    X = muestras_cercanas(inst, r["precios"], 20_000, 2, escala=0.03)
    factibles = violacion(inst, X) <= TOL_FACTIBLE
    assert factibles.sum() > 100 and utilidad(inst, X[factibles]).max() < r["utilidad"]


@prueba("06", "M03: hipercubo latino")
def _():
    inst = INST_PRUEBA_F
    P = etapa_1_inicializar_M03(40, inst, np.random.default_rng(3))
    U = (P - inst.piso) / (inst.techo - inst.piso)
    assert np.all((U >= 0) & (U < 1))
    assert all(sorted(np.floor(U[:, j] * 40).astype(int)) == list(range(40)) for j in range(inst.d))
    assert np.array_equal(P, etapa_1_inicializar_M03(40, inst, np.random.default_rng(3)))


@prueba("07", "M08: prioridad de factibilidad frente a penalización")
def _():
    ev = {"V": np.array([0.0, 0.0, 0.5, 0.1]), "utilidad": np.array([100.0, 120.0, 500.0, 50.0])}
    assert list(orden_M08(ev)) == [1, 0, 3, 2]
    penalizada = -ev["utilidad"] + 100.0 * ev["V"]           # M07 con un coeficiente bajo
    assert int(np.argmin(penalizada)) == 2 and ev["V"][2] > 0


@prueba("08", "M12: torneo y desempate")
def _():
    rangos = rangos_densos_M08({"V": np.zeros(3), "utilidad": np.array([9.0, 5.0, 1.0])})   # A, B, C
    assert list(rangos) == [0, 1, 2]
    assert torneo_desde_participantes(rangos, np.array([[2, 0, 2]]))[0] == 0
    empate = rangos_densos_M08({"V": np.zeros(3), "utilidad": np.array([7.0, 7.0, 1.0])})
    assert list(empate) == [0, 0, 1] and torneo_desde_participantes(empate, np.array([[1, 0]]))[0] == 1


@prueba("09", "M14: cruce por máscara")
def _():
    a, b = np.array([[1.0, 2, 3, 4]]), np.array([[5.0, 6, 7, 8]])
    h1, h2 = cruce_con_mascara(a, b, np.array([[1, 0, 1, 0]]))
    assert h1.tolist() == [[1, 6, 3, 8]] and h2.tolist() == [[5, 2, 7, 4]]
    A, B = np.random.default_rng(4).random((2, 200, 6))
    H1, H2 = etapa_4_recombinar_M14(A, B, np.random.default_rng(5))
    assert np.all((H1 == A) | (H1 == B)) and np.allclose(H1 + H2, A + B)


@prueba("10", "M15: segmento, suma y factibilidad por convexidad")
def _():
    a, b = np.array([[2.0, 10.0]]), np.array([[6.0, 2.0]])
    h1, h2 = combinacion_aritmetica(a, b, 0.25)
    assert np.allclose(h1, [[5.0, 4.0]]) and np.allclose(h2, [[3.0, 8.0]])
    inst = INST_PRUEBA
    X = muestras_factibles(inst, 2000, 6)
    H1, H2 = etapa_4_recombinar_M15(X[:1000], X[1000:], np.random.default_rng(7))
    assert np.allclose(H1 + H2, X[:1000] + X[1000:])
    assert np.all((H1 >= np.minimum(X[:1000], X[1000:]) - 1e-12) & (H1 <= np.maximum(X[:1000], X[1000:]) + 1e-12))
    assert violacion(inst, np.vstack([H1, H2])).max() <= 1e-12, "un hijo de padres factibles resultó inviable"


@prueba("11", "M16: BLX-α")
def _():
    inst = INST_PRUEBA
    rng = np.random.default_rng(8)
    A = inst.piso + rng.random((3000, inst.d)) * (inst.techo - inst.piso)
    B = inst.piso + rng.random((3000, inst.d)) * (inst.techo - inst.piso)
    bajo, alto, I = np.minimum(A, B), np.maximum(A, B), np.abs(A - B)
    H1, _h = etapa_4_recombinar_M16(A, B, rng, 0.0, inst.piso, inst.techo)
    assert np.all((H1 >= bajo) & (H1 <= alto))
    H1, H2 = etapa_4_recombinar_M16(A, B, rng, 0.5, inst.piso, inst.techo)
    for H in (H1, H2):
        assert np.all((H >= np.maximum(bajo - 0.5 * I, inst.piso)) & (H <= np.minimum(alto + 0.5 * I, inst.techo)))
    assert np.mean((H1 == inst.piso) | (H1 == inst.techo)) > 0, "el recorte debería dejar masa en las cotas"
    H1, H2 = etapa_4_recombinar_M16(A, A, rng, 0.5, inst.piso, inst.techo)
    assert np.array_equal(H1, A) and np.array_equal(H2, A)


@prueba("12", "M18: reinicio uniforme")
def _():
    inst = INST_PRUEBA
    H = inst.piso + np.random.default_rng(9).random((4000, inst.d)) * (inst.techo - inst.piso)
    copia = H.copy()
    assert np.array_equal(etapa_5_mutar_M18(H, 0.0, np.random.default_rng(10), inst.piso, inst.techo), H)
    Y = etapa_5_mutar_M18(H, 1.0, np.random.default_rng(10), inst.piso, inst.techo)
    assert np.array_equal(H, copia) and np.all((Y >= inst.piso) & (Y <= inst.techo)) and not np.any(Y == H)
    centro, ee = (inst.piso + inst.techo) / 2, (inst.techo - inst.piso) / np.sqrt(12 * len(Y))
    assert np.all(np.abs(Y.mean(axis=0) - centro) < 4 * ee)


@prueba("13", "M19: gaussiana acotada")
def _():
    inst = INST_PRUEBA
    R, centro = inst.techo - inst.piso, (inst.piso + inst.techo) / 2
    n = 20_000
    Y = etapa_5_mutar_M19(np.tile(centro, (n, 1)), 1.0, np.random.default_rng(11), inst.piso, inst.techo, 0.05)
    assert np.all((Y >= inst.piso) & (Y <= inst.techo))
    assert np.all(np.abs(Y.mean(axis=0) - centro) < 4 * 0.05 * R / np.sqrt(n))
    assert np.all(np.abs(Y.std(axis=0) / (0.05 * R) - 1) < 0.03)
    Y = etapa_5_mutar_M19(np.tile(inst.techo, (n, 1)), 1.0, np.random.default_rng(12), inst.piso, inst.techo, 0.05)
    assert np.all(np.abs(np.mean(Y == inst.techo, axis=0) - 0.5) < 0.02)


@prueba("14", "M20: polinomial acotada")
def _():
    inst = INST_PRUEBA
    centro = (inst.piso + inst.techo) / 2
    n = 200_000 // 3
    for x in (inst.piso, inst.techo, centro):
        for eta in (5, 20, 50):
            Y = etapa_5_mutar_M20(np.tile(x, (n, 1)), 1.0, np.random.default_rng(13), inst.piso, inst.techo, eta)
            assert np.all((Y >= inst.piso) & (Y <= inst.techo))
    rng = np.random.default_rng(14)
    Y5 = etapa_5_mutar_M20(np.tile(centro, (n, 1)), 1.0, rng, inst.piso, inst.techo, 5)
    Y50 = etapa_5_mutar_M20(np.tile(centro, (n, 1)), 1.0, rng, inst.piso, inst.techo, 50)
    assert np.all(Y5.std(axis=0) > Y50.std(axis=0))
    assert np.all(np.abs(Y5.mean(axis=0) - centro) < 4 * Y5.std(axis=0) / np.sqrt(n))


@prueba("15", "Integración: presupuesto, emparejamiento, reproducibilidad y elitismo")
def _():
    inst = INST_PRUEBA_F
    par = {"alfa": 0.3, "sigma": 0.05, "eta": 20}
    iniciales = [algoritmo_genetico(inst, c, m, 21, par, E_max=100)["poblacion"] for c, m in CONFIGURACIONES]
    assert all(np.array_equal(iniciales[0], P) for P in iniciales[1:]), "la población inicial debe ser común"
    for c, m in CONFIGURACIONES:
        r = algoritmo_genetico(inst, c, m, 22, par, E_max=3000)
        assert r["evaluaciones"] == 100 + r["generaciones"] * 98 and 3000 - 98 < r["evaluaciones"] <= 3000
        serie = [h["mejor_factible"] for h in r["historial"] if not np.isnan(h["mejor_factible"])]
        assert all(b >= a for a, b in zip(serie, serie[1:])), "la mejor utilidad factible no debe bajar"
    r1 = algoritmo_genetico(inst, "M16", "M20", 23, par, E_max=3000)
    r2 = algoritmo_genetico(inst, "M16", "M20", 23, par, E_max=3000)
    assert np.array_equal(r1["precios"], r2["precios"]) and r1["utilidad"] == r2["utilidad"]
    P = np.arange(3.0)[:, None]
    ev = {"V": np.zeros(3), "utilidad": np.array([5.0, 9.0, 7.0]), "factible": np.ones(3, bool)}
    evH = {"V": np.zeros(2), "utilidad": np.array([8.0, 1.0]), "factible": np.ones(2, bool)}
    P2, ev2 = etapa_6_reemplazar_M22(P, ev, np.array([[10.0], [11.0]]), evH, 1)
    assert ev2["utilidad"].tolist() == [9.0, 8.0, 1.0] and P2[:, 0].tolist() == [1.0, 10.0, 11.0]
    assert etapa_7_parar_M26(19_994, 98, 20_000) and not etapa_7_parar_M26(19_896, 98, 20_000)


@prueba("16", "Líneas base")
def _():
    inst = INST_PRUEBA_F
    r = busqueda_aleatoria(inst, 31, 3000)
    assert r["evaluaciones"] == 100 + 29 * 98
    p = ajuste_parejo(inst)
    assert violacion(inst, p["precios"])[0] <= TOL_FACTIBLE and p["utilidad"] >= utilidad(inst, inst.precio_actual)[0]
    assert 0.0 in np.round(np.arange(-0.25, ALZA_MAXIMA + 0.00025, 0.0005), 4)


tabla_pruebas = pd.DataFrame(RESULTADOS_PRUEBAS)
assert len(tabla_pruebas) == 16
print(tabla_pruebas.to_string(index=False))

prueba                                                           descripcion resultado
    01                                              Validación de instancias      pasó
    02                          Evaluación vectorizada idéntica a la escalar      pasó
    03                                           Un producto resuelto a mano      pasó
    04                            Dos sustitutos simétricos resueltos a mano      pasó
    05                             Certificado KKT en una instancia generada      pasó
    06                                                 M03: hipercubo latino      pasó
    07                  M08: prioridad de factibilidad frente a penalización      pasó
    08                                               M12: torneo y desempate      pasó
    09                                                M14: cruce por máscara      pasó
    10                     M15: segmento, suma y factibilidad por convexidad      pasó
    11                                     

## F. Instancias del estudio

Cinco instancias. La de **calibración**, C24-F, sólo se usa para elegir los parámetros de los operadores, para no ajustarlos con los mismos datos con que después se juzgan. Las cuatro de **evaluación** cruzan dos tamaños (12 y 36 productos) con las dos intensidades de sustitución. La instancia principal, **E12-F**, lleva nombres de productos de una categoría de bebidas para que los precios puedan leerse.

Para cada instancia se calcula el óptimo certificado, las dos líneas base deterministas y dos medidas de dificultad: la **fracción factible** de 20 000 vectores de precios sorteados al azar en la caja, y el **número de condición** de la curvatura en coordenadas normalizadas, $\kappa(R\,B\,R)$ con $R=\operatorname{diag}(u-\ell)$, que mide qué tan alargado es el valle que el algoritmo genético tiene que recorrer.

In [17]:
FAMILIAS_PRINCIPAL = ("Agua natural 1.5 L", "Refresco de cola 600 mL", "Jugo de naranja 1 L", "Bebida isotónica 600 mL")
DEFINICION = {
    "C24-F": dict(n_familias=8, intensidad="F", semilla=20261090, uso="calibración"),
    "E12-D": dict(n_familias=4, intensidad="D", semilla=20261091, uso="evaluación"),
    "E12-F": dict(n_familias=4, intensidad="F", semilla=20261092, uso="evaluación y caso principal",
                  familias=FAMILIAS_PRINCIPAL, costos_base=(11.0, 13.0, 22.0, 16.0),
                  volumenes=(3000, 3500, 1000, 1500)),
    "E36-D": dict(n_familias=12, intensidad="D", semilla=20261093, uso="evaluación"),
    "E36-F": dict(n_familias=12, intensidad="F", semilla=20261094, uso="evaluación"),
}
INSTANCIAS, EXACTOS, PAREJO = {}, {}, {}
filas = []
for nombre, definicion in DEFINICION.items():
    opciones = {k: v for k, v in definicion.items() if k != "uso"}
    inst = INSTANCIAS[nombre] = generar_instancia(nombre, **opciones)
    ex = EXACTOS[nombre] = resolver_exacto(inst)
    pa = PAREJO[nombre] = ajuste_parejo(inst)
    z0, zs = float(utilidad(inst, inst.precio_actual)[0]), ex["utilidad"]
    X = inst.piso + np.random.default_rng(0).random((20_000, inst.d)) * (inst.techo - inst.piso)
    R = np.diag(inst.techo - inst.piso)
    valores = np.linalg.eigvalsh(R @ inst.B @ R)
    tipos = [t for t, _ in ex["activas"]]
    filas.append({"instancia": nombre, "productos": inst.d, "sustitucion": definicion["intensidad"],
                  "uso": definicion["uso"], "utilidad_actual": z0, "utilidad_optima": zs,
                  "mejora_posible_pct": 100 * (zs - z0) / z0,
                  "ajuste_parejo_t_pct": 100 * pa["t"], "ajuste_parejo_utilidad": pa["utilidad"],
                  "brecha_actual_pct": 100 * (zs - z0) / zs, "brecha_parejo_pct": 100 * (zs - pa["utilidad"]) / zs,
                  "fraccion_factible_azar": float(np.mean(violacion(inst, X) <= TOL_FACTIBLE)),
                  "condicion": float(valores.max() / valores.min()),
                  "activas_existencias": tipos.count("existencias"), "activas_escalera": tipos.count("escalera"),
                  "activas_indice": tipos.count("índice"), "activas_techo": tipos.count("techo"),
                  "activas_piso": tipos.count("piso"), "activas_demanda": tipos.count("demanda"),
                  "residuo_kkt": ex["residuo"], "violacion_maxima": ex["max_violacion"],
                  "segundos_exacto": ex["segundos"]})
instancias = pd.DataFrame(filas)
instancias.to_csv(RESULTADOS / "instancias.csv", index=False)
with pd.option_context("display.width", 200, "display.max_columns", 30, "display.float_format", "{:,.4f}".format):
    print(instancias.drop(columns=["uso"]).to_string(index=False))

instancia  productos sustitucion  utilidad_actual  utilidad_optima  mejora_posible_pct  ajuste_parejo_t_pct  ajuste_parejo_utilidad  brecha_actual_pct  brecha_parejo_pct  fraccion_factible_azar  condicion  activas_existencias  activas_escalera  activas_indice  activas_techo  activas_piso  activas_demanda  residuo_kkt  violacion_maxima  segundos_exacto
    C24-F         24           F     166,257.4100     180,311.7128              8.4533               3.0000            176,614.5443             7.7944             2.0504                  0.0000    23.0386                    3                 0               1              0             0                0       0.0000            0.0000           0.0306
    E12-D         12           D      62,457.2500      66,004.5948              5.6796               3.0000             63,476.8459             5.3744             3.8297                  0.0004     8.5143                    2                 0               1              0             0    

## G. Diseño experimental e hipótesis

**Factores.** Recombinación (M14, M15, M16) × mutación (M18, M19, M20): nueve configuraciones en un factorial completo. Un diseño de un factor a la vez —variar la recombinación con la mutación fija y luego al revés— exigiría sólo cinco configuraciones, pero no podría detectar si la mejor recombinación depende de la mutación con que se acompaña.

**Fijo en las nueve.** $N=100$, $k=2$, $e=2$, $p_c=0.9$, $p_m=1/d$ y $E_{\max}=20\,000$, con las etapas M03, M08, M12, M22 y M26.

**Calibración (sección H).** Cada configuración calibra **sólo sus propios parámetros** en C24-F, con diez semillas (20261200–20261209): $\alpha\in\{0.1,0.3,0.5\}$ para M16, $\sigma\in\{0.02,0.05,0.10\}$ para M19 y $\eta_m\in\{5,20,50\}$ para M20. M16 con M19 o con M20 recorre las nueve parejas de valores; M14 y M15 con M18 no tienen nada que calibrar. En total son 35 celdas y 350 corridas. **Regla fijada antes de ejecutar:** se elige la celda de menor brecha mediana; un empate se rompe con la menor brecha media. Así ninguna forma compite con parámetros de otra ni con parámetros por defecto.

**Evaluación (sección I).** Cuatro instancias × nueve configuraciones × treinta semillas (20261017–20261046): 1 080 corridas, más 120 búsquedas aleatorias. Con la misma semilla, las nueve configuraciones parten de la misma población, de modo que **cada semilla es un bloque** y las comparaciones son emparejadas.

**Medidas.** La brecha respecto al óptimo certificado y la fracción de la mejora posible que se captura,
$$\text{brecha}=100\,\frac{\pi^\ast-\pi}{\pi^\ast},\qquad \text{mejora capturada}=100\,\frac{\pi-\pi_{\text{actual}}}{\pi^\ast-\pi_{\text{actual}}},$$
las evaluaciones hasta quedar a 1 % y a 0.1 % del óptimo, la dispersión de la población, la fracción de hijos factibles, la fracción de genes que quedan exactamente en una cota y la distancia entre los precios encontrados y los óptimos.

**Pruebas estadísticas (sección J).** Por instancia, la prueba de Friedman (1937) sobre las nueve configuraciones con las semillas como bloques, y la de rangos con signo de Wilcoxon (1945) de cada configuración contra la de menor rango medio, con la corrección de Holm (1979) para las ocho comparaciones. Los **efectos principales** se prueban con Friedman sobre las tres formas de una etapa, con bloques formados por la semilla y la forma de la otra etapa; los **efectos simples**, dentro de cada nivel del otro factor. El **ranking global** reúne 120 bloques (cuatro instancias × treinta semillas) y compara rangos medios con la prueba posterior de Demšar (2006) y Holm. El nivel de significancia es 0.05.

**Regla de elección final, fijada antes de ejecutar.** Se elige la configuración de menor rango medio global. Si alguna otra no difiere de ella según la prueba posterior con Holm, se elige, entre las que no difieren, la de **menor brecha máxima** en las 120 corridas: ante un empate estadístico se prefiere la que menos falla en su peor caso.

**Hipótesis.**

- **H1 · Mutación.** M18 será la peor forma de mutar en las cuatro instancias: reinicia el precio en todo su intervalo y no puede afinar cerca del óptimo. M19 y M20, que perturban localmente, quedarán cerca una de otra.
- **H2 · Recombinación.** M15 avanzará más rápido al principio, porque de padres factibles produce hijos factibles, pero contraerá la población y quedará atrás al final del presupuesto. M16 conservará más dispersión.
- **H3 · Interacción.** El orden de las tres recombinaciones cambiará según la mutación que las acompañe. En particular, con M18 la contracción de M15 compensará el ruido del reinicio.
- **H4 · Dificultad.** Las diferencias entre configuraciones crecerán con el tamaño y con la sustitución fuerte.

> **Transparencia.** Estas hipótesis se escribieron después de un piloto de diseño con parámetros por defecto ($\alpha=0.5$, $\sigma=0.05$, $\eta_m=20$), diez semillas y versiones preliminares de las instancias, que sirvió para fijar el presupuesto y las rejillas de calibración. No son predicciones a ciegas. El experimento formal usa instancias regeneradas con otras semillas, parámetros calibrados en una instancia aparte y semillas algorítmicas nuevas.

## H. Calibración de los parámetros propios en una instancia separada

In [18]:
E_MAX = 20_000
REJILLA = {"alfa": (0.1, 0.3, 0.5), "sigma": (0.02, 0.05, 0.10), "eta": (5, 20, 50)}
SEMILLAS_CALIBRACION = list(range(20261200, 20261210))


def celdas(cruce: str, mutacion: str) -> list:
    nombres = [p for p in (PARAMETRO_DE[cruce], PARAMETRO_DE[mutacion]) if p]
    return [dict(zip(nombres, valores)) for valores in product(*(REJILLA[n] for n in nombres))]


inst_c, z_c = INSTANCIAS["C24-F"], EXACTOS["C24-F"]["utilidad"]
inicio = time.perf_counter()
filas = []
for cruce, mutacion in CONFIGURACIONES:
    for par in celdas(cruce, mutacion):
        for s in SEMILLAS_CALIBRACION:
            r = algoritmo_genetico(inst_c, cruce, mutacion, s, par, E_max=E_MAX, historial=False)
            filas.append({"cruce": cruce, "mutacion": mutacion, "configuracion": f"{cruce}+{mutacion}",
                          "alfa": par.get("alfa", np.nan), "sigma": par.get("sigma", np.nan),
                          "eta": par.get("eta", np.nan), "semilla": s,
                          "brecha_pct": 100 * (z_c - r["utilidad"]) / z_c, "evaluaciones": r["evaluaciones"]})
calibracion = pd.DataFrame(filas)
calibracion.to_csv(RESULTADOS / "calibracion.csv", index=False)

PARAMETROS, elegidas = {}, []
claves = ["alfa", "sigma", "eta"]
celdas_c = (calibracion.groupby(["cruce", "mutacion"] + claves, dropna=False)["brecha_pct"]
            .agg(mediana="median", media="mean").reset_index())
for (cruce, mutacion), grupo in celdas_c.groupby(["cruce", "mutacion"]):
    g = grupo.sort_values(["mediana", "media"]).iloc[0]
    PARAMETROS[(cruce, mutacion)] = {k: (int(g[k]) if k == "eta" else float(g[k])) for k in claves if not pd.isna(g[k])}
    elegidas.append({"configuracion": f"{cruce}+{mutacion}", "celdas": len(grupo),
                     **{k: g[k] for k in claves}, "brecha_mediana_pct": g["mediana"], "brecha_media_pct": g["media"],
                     "segunda_mediana_pct": grupo.sort_values(["mediana", "media"])["mediana"].iloc[1] if len(grupo) > 1 else np.nan})
tabla_calibracion = pd.DataFrame(elegidas)
tabla_calibracion.to_csv(RESULTADOS / "calibracion_elegidos.csv", index=False)
print(f"{len(calibracion)} corridas de calibración en {time.perf_counter() - inicio:.1f} s")
print(tabla_calibracion.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

350 corridas de calibración en 6.9 s
configuracion  celdas   alfa  sigma     eta  brecha_mediana_pct  brecha_media_pct  segunda_mediana_pct
      M14+M18       1    NaN    NaN     NaN              1.2073            1.3023                  NaN
      M14+M19       3    NaN 0.0500     NaN              0.2780            0.3242               0.4574
      M14+M20       3    NaN    NaN 20.0000              0.4315            0.4639               0.5368
      M15+M18       1    NaN    NaN     NaN              0.9975            0.9829                  NaN
      M15+M19       3    NaN 0.1000     NaN              0.4415            0.4327               0.4866
      M15+M20       3    NaN    NaN 20.0000              0.4129            0.4244               0.5524
      M16+M18       3 0.3000    NaN     NaN              0.9511            0.9995               1.0523
      M16+M19       9 0.3000 0.0500     NaN              0.2438            0.2489               0.2726
      M16+M20       9 0.3000    NaN 

## I. Experimento factorial en las cuatro instancias de evaluación

Cada corrida usa los parámetros que su configuración eligió en la calibración. La trayectoria completa se guarda generación por generación. La búsqueda aleatoria de cada semilla usa exactamente las 19 994 evaluaciones de una corrida genética y se registra con la misma granularidad: 100 evaluaciones iniciales y bloques de 98.

In [19]:
INSTANCIAS_EVALUACION = ["E12-D", "E12-F", "E36-D", "E36-F"]
SEMILLAS = list(range(20261017, 20261047))
EVALUACIONES_AG = 100 + ((E_MAX - 100) // 98) * 98          # 19 994
GEN_25, GEN_50 = 50, 101                                      # 5 000 y 9 998 evaluaciones


def primera_evaluacion(evaluaciones: np.ndarray, brechas: np.ndarray, umbral: float) -> float:
    idx = np.flatnonzero(brechas <= umbral)
    return float(evaluaciones[idx[0]]) if len(idx) else np.nan


inicio = time.perf_counter()
filas, historiales, filas_ba, historiales_ba = [], [], [], []
for nombre in INSTANCIAS_EVALUACION:
    inst, ex = INSTANCIAS[nombre], EXACTOS[nombre]
    zs, z0 = ex["utilidad"], float(utilidad(inst, inst.precio_actual)[0])
    for cruce, mutacion in CONFIGURACIONES:
        par = PARAMETROS[(cruce, mutacion)]
        for s in SEMILLAS:
            r = algoritmo_genetico(inst, cruce, mutacion, s, par, E_max=E_MAX)
            h = pd.DataFrame(r["historial"])
            h["brecha_pct"] = 100 * (zs - h["mejor_factible"]) / zs
            ev_h, br_h = h["evaluaciones"].to_numpy(), h["brecha_pct"].to_numpy()
            filas.append({"instancia": nombre, "cruce": cruce, "mutacion": mutacion,
                          "configuracion": f"{cruce}+{mutacion}", "semilla": s,
                          "parametros": json.dumps(par), "utilidad": r["utilidad"],
                          "brecha_pct": 100 * (zs - r["utilidad"]) / zs,
                          "mejora_capturada_pct": 100 * (r["utilidad"] - z0) / (zs - z0),
                          "brecha_5000_pct": br_h[GEN_25], "brecha_9998_pct": br_h[GEN_50],
                          "eval_1pct": primera_evaluacion(ev_h, br_h, 1.0),
                          "eval_01pct": primera_evaluacion(ev_h, br_h, 0.1),
                          "eval_mejor": r["eval_mejor"], "evaluaciones": r["evaluaciones"],
                          "fraccion_factible_final": r["fraccion_factible"], "dispersion_final": r["dispersion"],
                          "en_cota_final": r["en_cota"],
                          "distancia_precios_pp": float(100 * np.max(np.abs(r["precios"] - ex["precios"]) / inst.precio_actual)),
                          "segundos": r["segundos"]})
            h = h.assign(instancia=nombre, cruce=cruce, mutacion=mutacion, semilla=s)
            historiales.append(h[["instancia", "cruce", "mutacion", "semilla", "generacion", "evaluaciones",
                                  "brecha_pct", "fraccion_factible", "fraccion_hijos_factibles", "dispersion", "en_cota"]])
    for s in SEMILLAS:
        ba = busqueda_aleatoria(inst, s, EVALUACIONES_AG)
        if ba["evaluaciones"] != EVALUACIONES_AG:
            raise RuntimeError("La búsqueda aleatoria no usó el presupuesto del AG.")
        filas_ba.append({"instancia": nombre, "semilla": s, "factible": ba["factible"], "utilidad": ba["utilidad"],
                         "brecha_pct": 100 * (zs - ba["utilidad"]) / zs if ba["factible"] else np.nan,
                         "violacion_minima": ba["violacion_minima"], "evaluaciones": ba["evaluaciones"]})
        t = pd.DataFrame(ba["trayectoria"]).assign(instancia=nombre, semilla=s)
        t["brecha_pct"] = 100 * (zs - t["mejor_factible"]) / zs
        historiales_ba.append(t[["instancia", "semilla", "evaluaciones", "brecha_pct"]])

corridas = pd.DataFrame(filas)
historial = pd.concat(historiales, ignore_index=True)
aleatoria = pd.DataFrame(filas_ba)
historial_ba = pd.concat(historiales_ba, ignore_index=True)
corridas.to_csv(RESULTADOS / "corridas.csv", index=False)
historial.to_csv(RESULTADOS / "historiales.csv", index=False, float_format="%.6g")
aleatoria.to_csv(RESULTADOS / "busqueda_aleatoria.csv", index=False)
historial_ba.to_csv(RESULTADOS / "historial_busqueda_aleatoria.csv", index=False, float_format="%.6g")
assert (corridas["evaluaciones"] == EVALUACIONES_AG).all() and (corridas["brecha_pct"] >= -1e-8).all()
print(f"{len(corridas)} corridas genéticas y {len(aleatoria)} búsquedas aleatorias en {time.perf_counter() - inicio:.1f} s")
print("Mediana por corrida genética:", f"{corridas['segundos'].median():.3f} s")

1080 corridas genéticas y 120 búsquedas aleatorias en 24.0 s
Mediana por corrida genética: 0.019 s


## J. Análisis estadístico

### J.1 Resumen por instancia y configuración

In [20]:
ORDEN_CONFIG = [f"{c}+{m}" for c, m in CONFIGURACIONES]


def q1(x):
    return x.quantile(0.25)


def q3(x):
    return x.quantile(0.75)


resumen = (corridas.groupby(["instancia", "configuracion", "cruce", "mutacion"])
           .agg(brecha_mediana=("brecha_pct", "median"), brecha_q1=("brecha_pct", q1), brecha_q3=("brecha_pct", q3),
                brecha_max=("brecha_pct", "max"), brecha_min=("brecha_pct", "min"),
                mejora_capturada_mediana=("mejora_capturada_pct", "median"),
                brecha_5000_mediana=("brecha_5000_pct", "median"), brecha_9998_mediana=("brecha_9998_pct", "median"),
                aciertos_01=("brecha_pct", lambda x: int((x <= 0.1).sum())),
                eval_1pct_mediana=("eval_1pct", "median"), eval_01pct_mediana=("eval_01pct", "median"),
                sin_01=("eval_01pct", lambda x: int(x.isna().sum())),
                dispersion_final=("dispersion_final", "median"), en_cota_final=("en_cota_final", "median"),
                distancia_precios_pp=("distancia_precios_pp", "median"), segundos=("segundos", "median"))
           .reset_index())
resumen["configuracion"] = pd.Categorical(resumen["configuracion"], ORDEN_CONFIG, ordered=True)
resumen = resumen.sort_values(["instancia", "configuracion"]).reset_index(drop=True)
resumen.to_csv(RESULTADOS / "resumen.csv", index=False)
tabla_3x3 = resumen.pivot(index=["instancia", "cruce"], columns="mutacion", values="brecha_mediana")
print("Brecha mediana (%) — filas: recombinación; columnas: mutación")
print(tabla_3x3.to_string(float_format=lambda v: f"{v:.4f}"))

Brecha mediana (%) — filas: recombinación; columnas: mutación
mutacion           M18    M19    M20
instancia cruce                     
E12-D     M14   0.3222 0.0326 0.0316
          M15   0.9112 0.1438 0.0580
          M16   0.7402 0.0417 0.0408
E12-F     M14   0.6205 0.0878 0.0744
          M15   0.8777 0.1897 0.1105
          M16   0.7268 0.0732 0.0787
E36-D     M14   0.7689 0.0786 0.0729
          M15   0.8903 0.2095 0.2999
          M16   0.7557 0.0879 0.0975
E36-F     M14   1.4078 0.5478 0.4744
          M15   0.9308 0.4606 0.6984
          M16   0.8407 0.3646 0.3306


### J.2 Friedman por instancia, comparaciones contra la mejor, efectos principales y efectos simples

In [21]:
def holm(pvalores) -> np.ndarray:
    """Valores p ajustados por el procedimiento escalonado de Holm (1979)."""
    p = np.asarray(pvalores, float)
    m, ajustados, acumulado = len(p), np.empty(len(p)), 0.0
    for i, j in enumerate(np.argsort(p)):
        acumulado = max(acumulado, min(1.0, (m - i) * p[j]))
        ajustados[j] = acumulado
    return ajustados


def wilcoxon_emparejado(x: np.ndarray, y: np.ndarray) -> float:
    return 1.0 if np.all(x == y) else float(wilcoxon(x, y).pvalue)


def comparar(M: np.ndarray, etiquetas: list) -> dict:
    """Friedman sobre las columnas de M (bloques en filas) y Wilcoxon-Holm de cada columna contra la de menor rango."""
    chi2, p = friedmanchisquare(*M.T)
    rangos = rankdata(M, axis=1).mean(axis=0)
    mejor = int(np.argmin(rangos))
    otros = [j for j in range(M.shape[1]) if j != mejor]
    p_crudos = [wilcoxon_emparejado(M[:, j], M[:, mejor]) for j in otros]
    p_holm = dict(zip(otros, holm(p_crudos)))
    return {"chi2": float(chi2), "p": float(p), "rangos": dict(zip(etiquetas, rangos)), "mejor": etiquetas[mejor],
            "p_holm": {etiquetas[j]: float(p_holm[j]) for j in otros},
            "empatadas": [etiquetas[j] for j in otros if p_holm[j] >= 0.05]}


def matriz_bloques(df: pd.DataFrame, indice, columna: str, niveles) -> np.ndarray:
    return df.pivot_table(index=indice, columns=columna, values="brecha_pct").loc[:, list(niveles)].to_numpy()


POR_INSTANCIA, PRINCIPALES, SIMPLES, filas_est = {}, {}, {}, []
for nombre in INSTANCIAS_EVALUACION:
    df = corridas[corridas["instancia"] == nombre]
    c = POR_INSTANCIA[nombre] = comparar(matriz_bloques(df, "semilla", "configuracion", ORDEN_CONFIG), ORDEN_CONFIG)
    filas_est.append({"instancia": nombre, "prueba": "Friedman, 9 configuraciones", "estadistico": c["chi2"],
                      "p": c["p"], "mejor": c["mejor"], "empatadas_con_la_mejor": " · ".join(c["empatadas"]) or "ninguna"})
    for factor, otro, niveles, otros_niveles in (("cruce", "mutacion", CRUCES, MUTACIONES),
                                                 ("mutacion", "cruce", MUTACIONES, CRUCES)):
        e = PRINCIPALES[(nombre, factor)] = comparar(matriz_bloques(df, ["semilla", otro], factor, niveles), list(niveles))
        filas_est.append({"instancia": nombre, "prueba": f"Efecto principal de {factor}", "estadistico": e["chi2"],
                          "p": e["p"], "mejor": e["mejor"], "empatadas_con_la_mejor": " · ".join(e["empatadas"]) or "ninguna"})
        for nivel in otros_niveles:
            sub = df[df[otro] == nivel]
            s_ = SIMPLES[(nombre, factor, nivel)] = comparar(matriz_bloques(sub, "semilla", factor, niveles), list(niveles))
            filas_est.append({"instancia": nombre, "prueba": f"Efecto simple de {factor} con {nivel}",
                              "estadistico": s_["chi2"], "p": s_["p"], "mejor": s_["mejor"],
                              "empatadas_con_la_mejor": " · ".join(s_["empatadas"]) or "ninguna"})
estadistica = pd.DataFrame(filas_est)
estadistica.to_csv(RESULTADOS / "pruebas_estadisticas.csv", index=False)
with pd.option_context("display.width", 200, "display.max_colwidth", 60):
    print(estadistica.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

instancia                            prueba  estadistico        p   mejor      empatadas_con_la_mejor
    E12-D       Friedman, 9 configuraciones          214 5.64e-42 M14+M20           M14+M19 · M16+M20
    E12-D         Efecto principal de cruce          114 1.42e-25     M14                     ninguna
    E12-D    Efecto simple de cruce con M18         42.5    6e-10     M14                     ninguna
    E12-D    Efecto simple de cruce con M19         48.3  3.3e-11     M14                     ninguna
    E12-D    Efecto simple de cruce con M20         28.5 6.58e-07     M14                     ninguna
    E12-D      Efecto principal de mutacion          141 1.95e-31     M20                     ninguna
    E12-D Efecto simple de mutacion con M14         45.3 1.48e-10     M20                         M19
    E12-D Efecto simple de mutacion con M15           60 9.36e-14     M20                     ninguna
    E12-D Efecto simple de mutacion con M16           45 1.69e-10     M19         

### J.3 Ranking global y elección final

Los 120 bloques (cuatro instancias × treinta semillas) se ordenan por rangos dentro de cada bloque, de modo que las instancias con brechas mayores no pesan más que las otras. La prueba posterior compara el rango medio de cada configuración con el de la mejor, con el estadístico $z=(R_j-R_{\text{mejor}})/\sqrt{k(k+1)/(6n)}$ (Demšar, 2006), $k=9$ y $n=120$, y los valores $p$ se ajustan con Holm.

In [22]:
M_global = np.vstack([matriz_bloques(corridas[corridas["instancia"] == n], "semilla", "configuracion", ORDEN_CONFIG)
                      for n in INSTANCIAS_EVALUACION])
chi2_g, p_g = friedmanchisquare(*M_global.T)
R_g = rankdata(M_global, axis=1).mean(axis=0)
k_g, n_g = M_global.shape[1], M_global.shape[0]
j_mejor = int(np.argmin(R_g))
ee_g = np.sqrt(k_g * (k_g + 1) / (6 * n_g))
otros = [j for j in range(k_g) if j != j_mejor]
p_crudo = [2 * (1 - norm.cdf(abs(R_g[j] - R_g[j_mejor]) / ee_g)) for j in otros]
p_ajustado = dict(zip(otros, holm(p_crudo)))
global_ = pd.DataFrame({"configuracion": ORDEN_CONFIG, "rango_medio": R_g,
                        "p_holm_vs_mejor": [p_ajustado.get(j, np.nan) for j in range(k_g)],
                        "brecha_maxima_pct": [corridas.loc[corridas["configuracion"] == c, "brecha_pct"].max() for c in ORDEN_CONFIG],
                        "brecha_mediana_pct": [corridas.loc[corridas["configuracion"] == c, "brecha_pct"].median() for c in ORDEN_CONFIG],
                        "gana_al_elegido": np.nan}).sort_values("rango_medio").reset_index(drop=True)

# Regla fijada en G: menor rango medio; si hay empates estadísticos, la de menor brecha máxima entre ellas.
candidatas = global_[(global_["configuracion"] == ORDEN_CONFIG[j_mejor]) | (global_["p_holm_vs_mejor"] >= 0.05)]
ELEGIDA = candidatas.sort_values(["brecha_maxima_pct", "rango_medio"]).iloc[0]["configuracion"]
j_el = ORDEN_CONFIG.index(ELEGIDA)
global_["gana_al_elegido"] = [float(np.mean(M_global[:, ORDEN_CONFIG.index(c)] < M_global[:, j_el])) for c in global_["configuracion"]]
global_["pierde_contra_el_elegido"] = [float(np.mean(M_global[:, ORDEN_CONFIG.index(c)] > M_global[:, j_el])) for c in global_["configuracion"]]
global_.to_csv(RESULTADOS / "ranking_global.csv", index=False)
print(f"Friedman global: chi2 = {chi2_g:.2f}, p = {p_g:.3g}, n = {n_g} bloques")
print(global_.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nConfiguración de menor rango medio:", ORDEN_CONFIG[j_mejor])
print("Configuraciones que no difieren de ella:", ", ".join(candidatas["configuracion"]))
print("Configuración elegida por la regla:", ELEGIDA, "· parámetros", PARAMETROS[tuple(ELEGIDA.split("+"))])

Friedman global: chi2 = 735.13, p = 1.95e-153, n = 120 bloques
configuracion  rango_medio  p_holm_vs_mejor  brecha_maxima_pct  brecha_mediana_pct  gana_al_elegido  pierde_contra_el_elegido
      M16+M19        2.592              NaN             0.6303             0.07774            0.525                     0.475
      M14+M20          2.7                1              1.804             0.07297             0.55                      0.45
      M16+M20        2.758                1              0.575             0.09075                0                         0
      M14+M19        2.992           0.7737              1.718             0.08371           0.5083                    0.4917
      M15+M20         5.05        1.428e-11               1.02              0.2252            0.125                     0.875
      M15+M19        5.167         1.63e-12             0.7567              0.2015             0.05                      0.95
      M16+M18          7.6                0            

## K. Mecanismos: dispersión, factibilidad de los hijos y recorte en las cotas

Las medidas de la población explican **por qué** una forma gana o pierde. Se resumen con la mediana de las treinta semillas en cada generación.

In [23]:
mecanismo = (historial.groupby(["instancia", "cruce", "mutacion", "generacion"])
             [["evaluaciones", "brecha_pct", "dispersion", "fraccion_hijos_factibles", "fraccion_factible", "en_cota"]]
             .median().reset_index())
mecanismo.to_csv(RESULTADOS / "mecanismo_medianas.csv", index=False, float_format="%.6g")
cortes = [1, 10, GEN_25, GEN_50, int(mecanismo["generacion"].max())]
vista = mecanismo[(mecanismo["instancia"] == "E36-F") & (mecanismo["generacion"].isin(cortes))]
print("E36-F: dispersión mediana de la población por generación")
print(vista.pivot_table(index=["cruce", "mutacion"], columns="generacion", values="dispersion").to_string(float_format=lambda v: f"{v:.4f}"))
print("\nE36-F: fracción mediana de hijos factibles por generación")
print(vista.pivot_table(index=["cruce", "mutacion"], columns="generacion", values="fraccion_hijos_factibles").to_string(float_format=lambda v: f"{v:.3f}"))
print("\nFracción de genes exactamente en una cota al final (mediana sobre las cuatro instancias)")
print(resumen.groupby(["cruce", "mutacion"])["en_cota_final"].median().unstack().to_string(float_format=lambda v: f"{v:.4f}"))

E36-F: dispersión mediana de la población por generación
generacion        1      10     50     101    203
cruce mutacion                                   
M14   M18      0.2784 0.2045 0.1197 0.1047 0.1031
      M19      0.2783 0.1932 0.0378 0.0299 0.0266
      M20      0.2782 0.1851 0.0441 0.0344 0.0273
M15   M18      0.2362 0.0716 0.0610 0.0629 0.0605
      M19      0.2352 0.0335 0.0231 0.0209 0.0207
      M20      0.2349 0.0262 0.0167 0.0126 0.0121
M16   M18      0.2640 0.1320 0.0838 0.0820 0.0798
      M19      0.2639 0.1080 0.0216 0.0191 0.0173
      M20      0.2640 0.1083 0.0244 0.0201 0.0185

E36-F: fracción mediana de hijos factibles por generación
generacion       1     10    50    101   203
cruce mutacion                              
M14   M18      0.000 0.000 0.199 0.265 0.260
      M19      0.000 0.000 0.811 0.750 0.719
      M20      0.000 0.000 0.791 0.730 0.709
M15   M18      0.000 0.602 0.480 0.449 0.449
      M19      0.000 0.929 0.786 0.724 0.724
      M20      0.00

## L. Caso principal: los precios de las doce bebidas

Se comparan, producto por producto, los precios actuales, el óptimo certificado, la corrida de la configuración elegida con la primera semilla de la evaluación y el ajuste parejo. También se mide qué tan plana es la utilidad cerca del óptimo: cuánto cuesta **redondear** los precios óptimos a múltiplos de 10 y de 50 centavos, como se hace en un anaquel. Esa cifra da una escala práctica para leer las brechas del algoritmo genético.

In [24]:
inst_p, ex_p = INSTANCIAS["E12-F"], EXACTOS["E12-F"]
cruce_el, mutacion_el = ELEGIDA.split("+")
corrida_p = algoritmo_genetico(inst_p, cruce_el, mutacion_el, SEMILLAS[0], PARAMETROS[(cruce_el, mutacion_el)], E_max=E_MAX)
peor = global_.iloc[-1]["configuracion"]
corrida_peor = algoritmo_genetico(inst_p, *peor.split("+"), SEMILLAS[0], PARAMETROS[tuple(peor.split("+"))], E_max=E_MAX)
z0_p, zs_p = float(utilidad(inst_p, inst_p.precio_actual)[0]), ex_p["utilidad"]
activa_de = {}
for (tipo, j), lam in zip(ex_p["activas"], ex_p["multiplicadores"]):
    if j >= 0:
        activa_de[j] = f"{tipo} (λ = {lam:,.0f})"
q_opt = inst_p.demanda(ex_p["precios"])
caso = pd.DataFrame({
    "producto": [inst_p.producto(j) for j in range(inst_p.d)],
    "costo": inst_p.costo, "piso": inst_p.piso, "precio_actual": inst_p.precio_actual, "techo": inst_p.techo,
    "precio_optimo": ex_p["precios"], "cambio_optimo_pct": 100 * (ex_p["precios"] / inst_p.precio_actual - 1),
    "precio_ag": corrida_p["precios"], "dif_ag_pesos": corrida_p["precios"] - ex_p["precios"],
    "precio_parejo": PAREJO["E12-F"]["precios"],
    "demanda_actual": inst_p.demanda_actual, "demanda_optima": q_opt, "existencias": inst_p.existencias,
    "regla_activa": [activa_de.get(j, "") for j in range(inst_p.d)]})
caso.to_csv(RESULTADOS / "caso_principal_productos.csv", index=False)

alternativas = {"Precios actuales (costo más 45 %)": inst_p.precio_actual,
                f"Ajuste parejo ({100 * PAREJO['E12-F']['t']:+.2f} %)": PAREJO["E12-F"]["precios"],
                f"AG {ELEGIDA}, semilla {SEMILLAS[0]}": corrida_p["precios"],
                f"AG {peor}, semilla {SEMILLAS[0]}": corrida_peor["precios"],
                "Óptimo redondeado a 50 centavos": np.round(ex_p["precios"] * 2) / 2,
                "Óptimo redondeado hacia abajo a 50 centavos": np.floor(ex_p["precios"] * 2) / 2,
                "Óptimo redondeado a 10 centavos": np.round(ex_p["precios"], 1),
                "Óptimo certificado": ex_p["precios"]}
filas = []
for nombre, p in alternativas.items():
    u = float(utilidad(inst_p, p)[0])
    filas.append({"alternativa": nombre, "utilidad": u, "brecha_pct": 100 * (zs_p - u) / zs_p,
                  "mejora_capturada_pct": 100 * (u - z0_p) / (zs_p - z0_p),
                  "factible": bool(violacion(inst_p, p)[0] <= TOL_FACTIBLE), "violacion": float(violacion(inst_p, p)[0]),
                  "indice_precios": float(inst_p.pesos_indice @ (p / inst_p.precio_actual)),
                  "reglas_violadas": "; ".join(f"{t} {inst_p.producto(j) if j >= 0 else ''}".strip()
                                               for (t, j), g in zip(inst_p.tipo, inst_p.G @ p - inst_p.h) if g > TOL_FACTIBLE)})
caso_alternativas = pd.DataFrame(filas)
caso_alternativas.to_csv(RESULTADOS / "caso_principal_alternativas.csv", index=False)
print("Reglas activas en el óptimo y sus multiplicadores:",
      "; ".join(f"{t} {inst_p.producto(j) if j >= 0 else ''}".strip() + f": λ = {lam:,.2f}"
                for (t, j), lam in zip(ex_p["activas"], ex_p["multiplicadores"])))
for nombre in INSTANCIAS:
    print(f"  {nombre}:", "; ".join(f"{t}{'' if j < 0 else ' ' + str(j)} λ = {lam:,.1f}"
                                    for (t, j), lam in zip(EXACTOS[nombre]["activas"], EXACTOS[nombre]["multiplicadores"])))
print(caso_alternativas.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print(caso[["producto", "precio_actual", "precio_optimo", "cambio_optimo_pct", "precio_ag", "dif_ag_pesos",
                "demanda_optima", "existencias", "regla_activa"]].to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

Reglas activas en el óptimo y sus multiplicadores: índice: λ = 128,639.59
  C24-F: existencias 3 λ = 539.0; existencias 6 λ = 1,918.8; existencias 19 λ = 1,222.5; índice λ = 372,930.4
  E12-D: existencias 3 λ = 428.8; existencias 9 λ = 49.2; índice λ = 47,528.6
  E12-F: índice λ = 128,639.6
  E36-D: existencias 18 λ = 21.6; índice λ = 170,480.7
  E36-F: existencias 3 λ = 93.1; existencias 9 λ = 357.4; índice λ = 552,723.9
                                alternativa    utilidad  brecha_pct  mejora_capturada_pct  factible  violacion  indice_precios                                              reglas_violadas
          Precios actuales (costo más 45 %) 55,502.9600      7.9802                0.0000      True     0.0000          1.0000                                                             
                    Ajuste parejo (+3.00 %) 59,131.0546      1.9651               75.3758      True     0.0000          1.0300                                                             
          

## M. Figuras

Todas se generan con los datos de esta ejecución y se guardan en `figuras/` como PNG a 200 ppp.

In [25]:
def guardar(fig, nombre: str):
    ruta = FIGURAS / nombre
    fig.savefig(ruta)
    plt.close(fig)
    print("Guardada:", ruta)


MILES = matplotlib.ticker.FuncFormatter(lambda v, _: "0" if v == 0 else f"{v / 1000:.0f} mil")
DECIMAL = matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}")


def eje_log_legible(ax, marcas=(0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10, 20)):
    """Escala logarítmica con marcas decimales legibles en lugar de potencias de diez."""
    ax.set_yscale("log")
    bajo, alto = ax.get_ylim()
    ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([m for m in marcas if bajo <= m <= alto]))
    ax.yaxis.set_major_formatter(DECIMAL)
    ax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())


def manijas_factoriales():
    manijas = [Line2D([], [], color=COLOR_CRUCE[c], lw=2.4, label=NOMBRE[c]) for c in CRUCES]
    manijas += [Line2D([], [], color=TINTA_2, lw=1.6, ls=TRAZO_MUTACION[m], marker=MARCA_MUTACION[m], ms=5,
                       label=NOMBRE[m]) for m in MUTACIONES]
    return manijas


def leyenda_factorial(ax, **kw):
    """Leyenda compacta: color = recombinación, trazo y marcador = mutación."""
    manijas = [Line2D([], [], color=COLOR_CRUCE[c], lw=2.4, label=NOMBRE[c]) for c in CRUCES]
    manijas += [Line2D([], [], color=TINTA_2, lw=1.6, ls=TRAZO_MUTACION[m], marker=MARCA_MUTACION[m], ms=5,
                       label=NOMBRE[m]) for m in MUTACIONES]
    return ax.legend(handles=manijas, **kw)


# Figura 1 — las siete etapas y el diseño factorial
fig = plt.figure(figsize=(11.5, 4.9))
ax = fig.add_axes([0, 0, 0.60, 1]); ax.set_axis_off(); ax.set_xlim(-0.6, 8.6); ax.set_ylim(0, 10)
etapas = [("1 · Inicialización", "M03 hipercubo latino", False), ("2 · Evaluación", "M08 prioridad de factibilidad", False),
          ("3 · Selección", "M12 torneo, k = 2", False), ("4 · Recombinación", "M14 · M15 · M16", True),
          ("5 · Mutación", "M18 · M19 · M20", True), ("6 · Reemplazo", "M22 elitista, e = 2", False),
          ("7 · Paro", "M26 presupuesto de 20 000", False)]
for i, (titulo, metodo, comparada) in enumerate(etapas):
    y = 9.0 - i * 1.25
    color_borde, fondo = (AZUL, "#e9f1fc") if comparada else (EJE, "white")
    ax.add_patch(FancyBboxPatch((0.6, y - 0.45), 5.6, 0.9, boxstyle="round,pad=0.02,rounding_size=0.18",
                                fc=fondo, ec=color_borde, lw=1.6 if comparada else 1.0))
    ax.text(0.85, y + 0.12, titulo, fontsize=10, color=TINTA, weight="bold" if comparada else "normal", va="center")
    ax.text(0.85, y - 0.2, metodo, fontsize=9, color=TINTA_2, va="center")
    if i < len(etapas) - 1:
        ax.add_patch(FancyArrowPatch((3.4, y - 0.47), (3.4, y - 0.78), arrowstyle="-|>", mutation_scale=9, color=TENUE, lw=1))
y_de = lambda i: 9.0 - i * 1.25
ax.add_patch(FancyArrowPatch((6.22, y_de(4)), (6.22, y_de(1)), connectionstyle="arc3,rad=0.55",
                             arrowstyle="-|>", mutation_scale=10, color=AZUL, lw=1.2))
ax.text(7.55, y_de(2.5), "los hijos se\nevalúan con\nla etapa 2\nantes del\nreemplazo", fontsize=8.5, color=TINTA_2,
        ha="center", va="center")
ax.add_patch(FancyArrowPatch((0.58, y_de(6)), (0.58, y_de(2)), connectionstyle="arc3,rad=-0.5",
                             arrowstyle="-|>", mutation_scale=10, color=TENUE, lw=1.0))
ax.text(0.05, y_de(4), "otra\ngeneración\nsi cabe en el\npresupuesto", fontsize=8, color=TINTA_2, ha="center",
        va="center", rotation=90)
ax.text(0.6, 9.85, "Etapas fijas en gris; las dos comparadas, en azul", fontsize=9, color=TINTA_2)
ax2 = fig.add_axes([0.63, 0.12, 0.33, 0.74])
ax2.set_xlim(-0.5, 2.5); ax2.set_ylim(2.5, -0.5); ax2.grid(False)
for i, c in enumerate(CRUCES):
    for j, m in enumerate(MUTACIONES):
        ax2.add_patch(Rectangle((j - 0.46, i - 0.46), 0.92, 0.92, fc="white", ec=COLOR_CRUCE[c], lw=1.6))
        ax2.text(j, i - 0.08, f"{c}+{m}", ha="center", va="center", fontsize=9.5, color=TINTA)
        ax2.plot(j, i + 0.22, marker=MARCA_MUTACION[m], color=COLOR_CRUCE[c], ms=6)
ax2.set_xticks(range(3)); ax2.set_xticklabels([NOMBRE[m] for m in MUTACIONES], fontsize=8.5)
ax2.set_yticks(range(3)); ax2.set_yticklabels([NOMBRE[c] for c in CRUCES], fontsize=8.5)
ax2.tick_params(length=0); ax2.xaxis.tick_top()
for lado in ax2.spines.values():
    lado.set_visible(False)
ax2.set_title("Diseño factorial 3 × 3: nueve configuraciones", fontsize=10, pad=34)
guardar(fig, "ae_a04_01_diseno_factorial.png")

Guardada: figuras/ae_a04_01_diseno_factorial.png


In [26]:
# Figura 2 — cómo genera hijos cada forma de recombinar y de mutar
rng_f = np.random.default_rng(20261016)
a_f, b_f = np.array([[0.30, 0.70]]), np.array([[0.62, 0.42]])
n_f = 4000
A_f, B_f = np.repeat(a_f, n_f, 0), np.repeat(b_f, n_f, 0)
cero, uno = np.zeros(2), np.ones(2)
hijos = {"M14": np.vstack(etapa_4_recombinar_M14(A_f, B_f, rng_f)),
         "M15": np.vstack(etapa_4_recombinar_M15(A_f, B_f, rng_f)),
         "M16": np.vstack(etapa_4_recombinar_M16(A_f, B_f, rng_f, 0.3, cero, uno))}
fig, ejes = plt.subplots(2, 3, figsize=(11.5, 7.2))
for ax, c in zip(ejes[0], CRUCES):
    H = hijos[c]
    if c == "M14":
        puntos, cuentas = np.unique(np.round(H, 6), axis=0, return_counts=True)
        ax.scatter(puntos[:, 0], puntos[:, 1], s=40 + cuentas / 12, color=COLOR_CRUCE[c], alpha=0.85, zorder=3)
        for (x, y), n in zip(puntos, cuentas):
            ax.annotate(f"{100 * n / len(H):.0f} %", (x, y), xytext=(7, -12), textcoords="offset points",
                        fontsize=8, color=TINTA_2)
    else:
        ax.scatter(H[:, 0], H[:, 1], s=3, color=COLOR_CRUCE[c], alpha=0.25, lw=0, zorder=2)
    ax.scatter([a_f[0, 0], b_f[0, 0]], [a_f[0, 1], b_f[0, 1]], s=70, marker="X", color=TINTA, zorder=4)
    ax.annotate("padre a", a_f[0], xytext=(-48, 6), textcoords="offset points", fontsize=8.5, color=TINTA)
    ax.annotate("padre b", b_f[0], xytext=(8, 6), textcoords="offset points", fontsize=8.5, color=TINTA)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("precio 1 (normalizado a su caja)"); ax.set_ylabel("precio 2")
    ax.set_title(NOMBRE[c] + (", α = 0.3" if c == "M16" else ""))
x_m = 0.88
n_m = 60_000
X_m = np.full((n_m, 1), x_m)
mutados = {"M18": etapa_5_mutar_M18(X_m, 1.0, rng_f, np.zeros(1), np.ones(1)),
           "M19": etapa_5_mutar_M19(X_m, 1.0, rng_f, np.zeros(1), np.ones(1), 0.10),
           "M20": etapa_5_mutar_M20(X_m, 1.0, rng_f, np.zeros(1), np.ones(1), 20)}
for ax, m in zip(ejes[1], MUTACIONES):
    y = mutados[m][:, 0]
    interior = y[y < 1]
    ax.hist(interior, bins=np.linspace(0, 1, 81), color=TINTA_2, alpha=0.75,
            weights=np.full(len(interior), 100 / n_m))
    en_techo = 100 * np.mean(y == 1)
    if en_techo > 0:
        ax.bar(1.0, en_techo, width=0.0125, color=NARANJA, align="edge")
        ax.annotate(f"{en_techo:.1f} % de las mutaciones\nqueda exactamente en el techo", (1.0, 0.8 * en_techo),
                    xytext=(0.12, 0.62), textcoords="axes fraction", fontsize=8, color=TINTA_2,
                    arrowprops=dict(arrowstyle="-", color=TINTA_2, lw=0.8))
    ax.axvline(x_m, color=TINTA, lw=1, ls="--")
    ax.text(x_m - 0.02, ax.get_ylim()[1] * 0.97, "gen original", ha="right", va="top", fontsize=8, color=TINTA,
            bbox=dict(fc="white", ec="none", pad=1))
    ax.set_xlim(0, 1.05); ax.set_xlabel("valor mutado (normalizado: 0 = piso, 1 = techo)")
    ax.set_ylabel("% de las mutaciones por intervalo")
    ax.set_title(NOMBRE[m] + {"M18": "", "M19": ", σ = 0.10", "M20": ", η = 20"}[m])
fig.tight_layout()
guardar(fig, "ae_a04_02_operadores.png")

Guardada: figuras/ae_a04_02_operadores.png


In [27]:
# Figura 3 — calibración en C24-F
cal = calibracion.groupby(["cruce", "mutacion", "alfa", "sigma", "eta"], dropna=False)["brecha_pct"].median().reset_index()
fig, ejes = plt.subplots(1, 5, figsize=(14.5, 3.6), gridspec_kw={"width_ratios": [1, 1, 1, 1.05, 1.05]})
for ax, (param, mut, cruces) in zip(ejes[:2], (("sigma", "M19", ("M14", "M15")), ("eta", "M20", ("M14", "M15")))):
    for c in cruces:
        g = cal[(cal["cruce"] == c) & (cal["mutacion"] == mut)].sort_values(param)
        ax.plot(range(3), g["brecha_pct"], color=COLOR_CRUCE[c], ls=TRAZO_MUTACION[mut], marker=MARCA_MUTACION[mut], ms=6, label=f"{c}+{mut}")
        el = PARAMETROS[(c, mut)][param]
        i = list(REJILLA[param]).index(el)
        ax.plot(i, g["brecha_pct"].iloc[i], marker="o", ms=13, mfc="none", mec=TINTA, lw=0)
    ax.set_xticks(range(3)); ax.set_xticklabels([str(v) for v in REJILLA[param]])
    ax.set_xlabel({"sigma": "σ de M19", "eta": "η de M20"}[param]); ax.set_ylabel("brecha mediana (%)")
    ax.legend(fontsize=8.5, loc="upper center")
g = cal[(cal["cruce"] == "M16") & (cal["mutacion"] == "M18")].sort_values("alfa")
ax = ejes[2]
ax.plot(range(3), g["brecha_pct"], color=COLOR_CRUCE["M16"], ls=TRAZO_MUTACION["M18"], marker=MARCA_MUTACION["M18"], ms=6, label="M16+M18")
i = list(REJILLA["alfa"]).index(PARAMETROS[("M16", "M18")]["alfa"])
ax.plot(i, g["brecha_pct"].iloc[i], marker="o", ms=13, mfc="none", mec=TINTA, lw=0)
ax.set_xticks(range(3)); ax.set_xticklabels([str(v) for v in REJILLA["alfa"]]); ax.set_xlabel("α de M16")
ax.set_ylabel("brecha mediana (%)"); ax.legend(fontsize=8.5, loc="upper left")
mapa_azul = LinearSegmentedColormap.from_list("azul", RAMPA_AZUL)
for ax, (param, mut) in zip(ejes[3:], (("sigma", "M19"), ("eta", "M20"))):
    g = cal[(cal["cruce"] == "M16") & (cal["mutacion"] == mut)]
    Z = g.pivot(index="alfa", columns=param, values="brecha_pct").loc[list(REJILLA["alfa"]), list(REJILLA[param])].to_numpy()
    im = ax.imshow(Z, cmap=mapa_azul, norm=LogNorm(vmin=Z.min(), vmax=Z.max()), aspect="auto")
    el = PARAMETROS[("M16", mut)]
    for r_ in range(3):
        for c_ in range(3):
            elegido = REJILLA["alfa"][r_] == el["alfa"] and REJILLA[param][c_] == el[param]
            oscuro = np.log(Z[r_, c_] / Z.min()) > 0.55 * np.log(Z.max() / Z.min())
            ax.text(c_, r_, f"{Z[r_, c_]:.3f}" + (" *" if elegido else ""), ha="center", va="center", fontsize=8.5,
                    color="white" if oscuro else TINTA, weight="bold" if elegido else "normal")
    ax.set_xticks(range(3)); ax.set_xticklabels([str(v) for v in REJILLA[param]])
    ax.set_yticks(range(3)); ax.set_yticklabels([str(v) for v in REJILLA["alfa"]])
    ax.set_xlabel({"sigma": "σ de M19", "eta": "η de M20"}[param]); ax.set_ylabel("α de M16")
    ax.set_title(f"M16+{mut}", fontsize=10); ax.grid(False)
for ax, t in zip(ejes, ("M14 y M15 con M19", "M14 y M15 con M20", "M16 con M18")):
    ax.set_title(t, fontsize=10)
fig.tight_layout()
guardar(fig, "ae_a04_03_calibracion.png")

Guardada: figuras/ae_a04_03_calibracion.png


In [28]:
# Figura 4 — mapa 3 × 3 de la brecha mediana por instancia
fig, ejes = plt.subplots(1, 4, figsize=(14.5, 3.9))
todas = resumen["brecha_mediana"]
norma = LogNorm(vmin=todas.min(), vmax=todas.max())
for ax, nombre in zip(ejes, INSTANCIAS_EVALUACION):
    r_ = resumen[resumen["instancia"] == nombre]
    Z = r_.pivot(index="cruce", columns="mutacion", values="brecha_mediana").loc[list(CRUCES), list(MUTACIONES)].to_numpy()
    ax.imshow(Z, cmap=mapa_azul, norm=norma, aspect="auto")
    c = POR_INSTANCIA[nombre]
    for i, cr in enumerate(CRUCES):
        for j, mu in enumerate(MUTACIONES):
            cfg = f"{cr}+{mu}"
            marca = " ●" if cfg == c["mejor"] else (" ≈" if cfg in c["empatadas"] else "")
            oscuro = norma(Z[i, j]) > 0.55
            ax.text(j, i, f"{Z[i, j]:.3f}{marca}", ha="center", va="center", fontsize=9,
                    color="white" if oscuro else TINTA, weight="bold" if cfg == c["mejor"] else "normal")
    ax.set_xticks(range(3)); ax.set_xticklabels(MUTACIONES); ax.set_yticks(range(3)); ax.set_yticklabels(CRUCES)
    ax.set_xlabel("mutación"); ax.set_ylabel("recombinación"); ax.grid(False)
    ax.set_title(f"{nombre}: óptimo {EXACTOS[nombre]['utilidad']:,.0f} $", fontsize=10)
fig.text(0.5, -0.03, "Brecha mediana (%) de treinta semillas. ● menor rango medio; ≈ no difiere de ella (Wilcoxon-Holm, α = 0.05). "
         "Escala de color logarítmica común.", ha="center", fontsize=9, color=TINTA_2)
fig.tight_layout()
guardar(fig, "ae_a04_04_brechas_3x3.png")

Guardada: figuras/ae_a04_04_brechas_3x3.png


In [29]:
# Figura 5 — interacción: brecha mediana por mutación, una línea por recombinación
fig, ejes = plt.subplots(1, 4, figsize=(14.5, 3.9), sharey=False)
for ax, nombre in zip(ejes, INSTANCIAS_EVALUACION):
    r_ = resumen[resumen["instancia"] == nombre]
    for desplazamiento, c in zip((-0.08, 0.0, 0.08), CRUCES):
        g = r_[r_["cruce"] == c].set_index("mutacion").loc[list(MUTACIONES)]
        x = np.arange(3) + desplazamiento
        ax.vlines(x, g["brecha_q1"], g["brecha_q3"], color=COLOR_CRUCE[c], lw=1.2, alpha=0.7)
        ax.plot(x, g["brecha_mediana"], color=COLOR_CRUCE[c], lw=2)
        for xi, mu, yi in zip(x, MUTACIONES, g["brecha_mediana"]):
            ax.plot(xi, yi, marker=MARCA_MUTACION[mu], color=COLOR_CRUCE[c], ms=7, mec="white", mew=1.2)
    eje_log_legible(ax); ax.set_xticks(range(3)); ax.set_xticklabels(MUTACIONES); ax.set_xlabel("mutación")
    ax.set_ylabel("brecha (%), escala log"); ax.set_title(nombre, fontsize=10)
manijas = [Line2D([], [], color=COLOR_CRUCE[c], lw=2, label=NOMBRE[c]) for c in CRUCES]
ejes[0].legend(handles=manijas, fontsize=8.5, loc="upper right")
fig.text(0.5, -0.03, "Punto: mediana de treinta semillas; barra: rango intercuartil. Si las líneas no son paralelas, "
         "la mejor recombinación depende de la mutación.", ha="center", fontsize=9, color=TINTA_2)
fig.tight_layout()
guardar(fig, "ae_a04_05_interaccion.png")

Guardada: figuras/ae_a04_05_interaccion.png


In [30]:
# Figura 6 — convergencia: brecha de la mejor solución factible frente a evaluaciones
med = historial.groupby(["instancia", "cruce", "mutacion", "generacion"]).agg(
    evaluaciones=("evaluaciones", "first"), brecha=("brecha_pct", "median")).reset_index()
med_ba = historial_ba.groupby(["instancia", "evaluaciones"])["brecha_pct"].median().reset_index()
fig, ejes = plt.subplots(1, 4, figsize=(14.5, 4.1))
for ax, nombre in zip(ejes, INSTANCIAS_EVALUACION):
    for c, m in CONFIGURACIONES:
        g = med[(med["instancia"] == nombre) & (med["cruce"] == c) & (med["mutacion"] == m)]
        ax.plot(g["evaluaciones"], g["brecha"], color=COLOR_CRUCE[c], ls=TRAZO_MUTACION[m], lw=1.6)
        ax.plot(g["evaluaciones"].iloc[-1], g["brecha"].iloc[-1], marker=MARCA_MUTACION[m], color=COLOR_CRUCE[c], ms=6)
    b = med_ba[med_ba["instancia"] == nombre].dropna()
    if len(b):
        ax.plot(b["evaluaciones"], b["brecha_pct"], color=TENUE, lw=1.4, ls="-.", label="búsqueda aleatoria")
    ax.axhline(instancias.set_index("instancia").loc[nombre, "brecha_parejo_pct"], color=TINTA, lw=0.9, ls=(0, (1, 2)))
    ax.text(EVALUACIONES_AG, instancias.set_index("instancia").loc[nombre, "brecha_parejo_pct"], "ajuste parejo ",
            ha="right", va="bottom", fontsize=8, color=TINTA)
    ax.set_yscale("log"); ax.set_ylim(max(1e-3, med[med["instancia"] == nombre]["brecha"].min() * 0.6), 25)
    eje_log_legible(ax)
    ax.set_xlabel("evaluaciones"); ax.set_ylabel("brecha mediana (%), escala log"); ax.set_title(nombre, fontsize=10)
    ax.set_xticks([0, 5000, 10000, 15000, 20000]); ax.xaxis.set_major_formatter(MILES)
manijas = manijas_factoriales() + [Line2D([], [], color=TENUE, lw=1.4, ls="-.", label="búsqueda aleatoria")]
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.legend(handles=manijas, loc="lower center", ncol=7, fontsize=8.5, bbox_to_anchor=(0.5, 0.0))
fig.text(0.5, -0.03, "Mediana de treinta semillas. La búsqueda aleatoria no aparece en E36-D ni en E36-F porque no encontró "
         "ninguna solución factible en ninguna semilla.", ha="center", fontsize=9, color=TINTA_2)
guardar(fig, "ae_a04_06_convergencia.png")

Guardada: figuras/ae_a04_06_convergencia.png


In [31]:
# Figura 7 — mecanismo en E36-F: dispersión, factibilidad de los hijos y velocidad frente a precisión
fig, ejes = plt.subplots(1, 3, figsize=(14.5, 4.3))
sub = mecanismo[mecanismo["instancia"] == "E36-F"]
for c, m in CONFIGURACIONES:
    g = sub[(sub["cruce"] == c) & (sub["mutacion"] == m)]
    ejes[0].plot(g["evaluaciones"], g["dispersion"], color=COLOR_CRUCE[c], ls=TRAZO_MUTACION[m], lw=1.5)
    g1 = g[g["generacion"] >= 1]
    ejes[1].plot(g1["evaluaciones"], g1["fraccion_hijos_factibles"], color=COLOR_CRUCE[c], ls=TRAZO_MUTACION[m], lw=1.5)
ejes[0].set_yscale("log"); eje_log_legible(ejes[0], marcas=(0.01, 0.02, 0.05, 0.1, 0.2, 0.3))
ejes[0].set_ylabel("dispersión D(P), escala log"); ejes[0].set_title("Dispersión de la población", fontsize=10)
ejes[1].set_ylabel("fracción de hijos factibles"); ejes[1].set_ylim(0, 1)
ejes[1].set_title("Hijos factibles por generación", fontsize=10)
for ax in ejes[:2]:
    ax.set_xlabel("evaluaciones"); ax.set_xticks([0, 5000, 10000, 15000, 20000]); ax.xaxis.set_major_formatter(MILES)
r36 = resumen[resumen["instancia"] == "E36-F"]
r36 = r36.sort_values("brecha_mediana")
posiciones = np.log10(r36["brecha_mediana"].to_numpy())
for i in range(1, len(posiciones)):                      # separa las etiquetas al menos 0.06 décadas
    posiciones[i] = max(posiciones[i], posiciones[i - 1] + 0.06)
for (_, fila), y_etq in zip(r36.iterrows(), posiciones):
    c, m = fila["cruce"], fila["mutacion"]
    ejes[2].plot([0, 1], [fila["brecha_5000_mediana"], fila["brecha_mediana"]], color=COLOR_CRUCE[c],
                 ls=TRAZO_MUTACION[m], lw=1.6, marker=MARCA_MUTACION[m], ms=6)
    ejes[2].text(1.05, 10 ** y_etq, fila["configuracion"], fontsize=7.5, va="center", color=TINTA_2)
eje_log_legible(ejes[2], marcas=(0.2, 0.3, 0.5, 1, 2, 3, 5))
ejes[2].set_xlim(-0.15, 1.35); ejes[2].set_xticks([0, 1])
ejes[2].set_xticklabels(["5 000\nevaluaciones", "19 994\nevaluaciones"])
ejes[2].set_ylabel("brecha mediana (%), escala log"); ejes[2].set_title("Al cuarto del presupuesto y al final", fontsize=10)
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.legend(handles=manijas_factoriales(), loc="lower center", ncol=6, fontsize=8.5, bbox_to_anchor=(0.5, 0.0))
guardar(fig, "ae_a04_07_mecanismo.png")

Guardada: figuras/ae_a04_07_mecanismo.png


In [32]:
# Figura 8 — caso principal: cambio de precio por producto
fig, ax = plt.subplots(figsize=(10.5, 5.8))
y = np.arange(inst_p.d)[::-1]
cambio_ag = 100 * (caso["precio_ag"] / caso["precio_actual"] - 1)
cambio_par = 100 * (caso["precio_parejo"] / caso["precio_actual"] - 1)
ax.barh(y, caso["cambio_optimo_pct"], height=0.62, color=AZUL, alpha=0.9, label="óptimo certificado")
ax.plot(cambio_ag, y, "D", color=NARANJA, ms=6.5, mec="white", mew=1, label=f"AG {ELEGIDA}, semilla {SEMILLAS[0]}")
ax.plot(cambio_par, y, "|", color=TINTA, ms=14, mew=2, label=f"ajuste parejo ({100 * PAREJO['E12-F']['t']:+.1f} %)")
ax.axvline(0, color=TINTA_2, lw=0.8)
ax.set_yticks(y); ax.set_yticklabels(caso["producto"], fontsize=8.5)
ax.set_xlabel("cambio respecto al precio actual (%)"); ax.grid(axis="y", visible=False)
ax.set_title("E12-F: precios de temporada de doce bebidas", fontsize=11)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.legend(loc="lower center", ncol=3, fontsize=8.5, bbox_to_anchor=(0.5, 0.0))
guardar(fig, "ae_a04_08_caso_principal.png")

Guardada: figuras/ae_a04_08_caso_principal.png


## N. Manifiesto de trazabilidad

Registra versiones, configuración, semillas y la huella SHA-256 de cada archivo de resultados y de cada figura. Dos ejecuciones con el mismo entorno deben producir huellas idénticas en todos los archivos salvo en las columnas de tiempo.

In [33]:
def sha256(ruta: Path) -> str:
    return hashlib.sha256(ruta.read_bytes()).hexdigest()


manifiesto = {
    "actividad": "AAEE · Actividad 04 · corte 2",
    "entorno": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                "scipy": scipy.__version__, "matplotlib": matplotlib.__version__, "plataforma": platform.platform()},
    "etapas_fijas": {"1": "M03", "2": "M08", "3": "M12 (k = 2)", "6": "M22 (e = 2)", "7": f"M26 (E_max = {E_MAX})"},
    "comparadas": {"4": list(CRUCES), "5": list(MUTACIONES)},
    "comunes": {"N": 100, "k": 2, "e": 2, "pc": 0.9, "pm": "1/d", "E_max": E_MAX, "evaluaciones": EVALUACIONES_AG},
    "rejilla_calibracion": REJILLA,
    "parametros_calibrados": {f"{c}+{m}": v for (c, m), v in PARAMETROS.items()},
    "semillas_calibracion": SEMILLAS_CALIBRACION, "semillas_evaluacion": SEMILLAS,
    "instancias": {n: {k: v for k, v in d_.items() if k not in ("familias", "costos_base", "volumenes")}
                   for n, d_ in DEFINICION.items()},
    "optimos": {n: e_["utilidad"] for n, e_ in EXACTOS.items()},
    "elegida": ELEGIDA,
    "pruebas": len(tabla_pruebas),
    "segundos_totales": time.perf_counter() - INICIO_CUADERNILLO,
    "archivos": {str(r): sha256(r) for r in sorted(list(RESULTADOS.glob("*.csv")) + list(FIGURAS.glob("*.png")))},
}
(RESULTADOS / "manifiesto.json").write_text(json.dumps(manifiesto, indent=2, ensure_ascii=False))
print(f"Manifiesto con {len(manifiesto['archivos'])} archivos · ejecución completa en {manifiesto['segundos_totales']:.0f} s")

Manifiesto con 21 archivos · ejecución completa en 33 s


## O. Lectura de resultados

**Referencias y líneas base.** Las cinco instancias tienen óptimo certificado: el residuo de estacionariedad es menor que $2\times10^{-9}$ y la violación máxima, menor que $2\times10^{-13}$. En todas, la regla activa decisiva es el **índice de precios**: el óptimo sube la categoría exactamente el 3 % permitido. En E12-F su multiplicador vale 128 640 pesos por unidad del índice, de modo que cada punto porcentual adicional de holgura valdría unos 1 286 pesos de utilidad por temporada. Las existencias limitan uno a tres productos en las demás instancias. La regla de costo más 45 % deja sobre la mesa entre 5.3 % y 8.7 % de la utilidad posible. El ajuste parejo, que sube todo el 3 %, recupera entre el 28 % y el 77 % de esa mejora. La búsqueda aleatoria con 19 994 evaluaciones encuentra soluciones factibles en las instancias de 12 productos, pero quedan a 7.0 % y 8.1 % del óptimo, peor o igual que no hacer nada. En las de 36 productos no encuentra **ninguna** solución factible en ninguna de las 30 semillas.

**Calibración.** Los parámetros elegidos quedaron en el interior de su rejilla salvo uno: $\alpha=0.3$ en las tres configuraciones con M16, $\eta_m=20$ en las tres con M20 y $\sigma=0.05$ con M14 y M16. La excepción es M15+M19, que eligió $\sigma=0.10$, el extremo de la rejilla: la recombinación que contrae necesita la mutación más amplia.

**Calidad final.** Las pruebas de Friedman rechazan la igualdad de las nueve configuraciones en las cuatro instancias ($p<10^{-34}$). Ninguna configuración es la mejor en todas: el menor rango medio corresponde a M14+M20 en E12-D, M16+M19 en E12-F, M14+M19 en E36-D y M16+M20 en E36-F, y sólo en E36-F la ganadora no tiene empates estadísticos. En el ranking global de 120 bloques, cuatro configuraciones forman el grupo superior y no difieren entre sí: M16+M19, M14+M20, M16+M20 y M14+M19, con rangos medios de 2.59 a 2.99. Las dos con M15 y mutación local quedan en 5.05 y 5.17, y las tres con M18, en 7.60 a 8.34.

**Hipótesis.**

- **H1 se confirma.** M18 es la peor mutación en las cuatro instancias, con cualquier recombinación. Sus mejores configuraciones capturan entre el 82 % y el 94 % de la mejora posible, contra el 91 % a 99 % de las demás. M19 y M20 empatan en E12-F y en E36-F; M20 gana en E12-D y M19 en E36-D.
- **H2 se confirma en el mecanismo y se refuta en la velocidad.** En E36-F, M15 produce más del 90 % de hijos factibles desde las primeras mil evaluaciones, mientras que M14 no produce ninguno hasta cerca de las tres mil, y M15 es la que más rápido contrae la población: en unas mil evaluaciones su dispersión cae a la décima parte de la inicial, cuando la de M14 apenas baja un tercio. Pero esa ventaja no se traduce en rapidez. A las 5 000 evaluaciones, M15 sólo va al frente en E36-F y sólo con M18. En las otras instancias van al frente M14 (E12-D) o M16 (E12-F y E36-D). Al final, M15 nunca entra al grupo superior.
- **H3 se sostiene en parte.** Hay interacción, pero es menor que los efectos principales. En E36-F, M15 supera a M14 con M18 y con M19, y queda detrás con M20. La mejor mutación para M15 depende del tamaño: M20 con 12 productos y M19 con 36. La predicción específica, que con M18 la contracción de M15 compensaría el reinicio, sólo se cumple en E36-F.
- **H4 se confirma en lo absoluto y cambia de forma.** Las brechas del grupo superior suben de 0.03–0.04 % en E12-D a 0.33–0.55 % en E36-F. Lo que más cambia es **qué recombinación gana**: en E12-D, E12-F y E36-D, M14 es la mejor o empata con la mejor; en la instancia grande con sustitución fuerte, M16 gana sola con las tres mutaciones.

**Elección final.** Por la regla fijada en G, entre las cuatro configuraciones que no difieren se elige la de menor brecha máxima: **M16+M20, con $\alpha=0.3$ y $\eta_m=20$**. Su peor corrida queda a 0.575 % del óptimo; las de M14+M19 y M14+M20 llegan a 1.72 % y 1.80 %. En mediana captura el 99.2 %, 99.0 %, 98.1 % y 95.8 % de la mejora posible en las cuatro instancias.

**Caso principal.** En E12-F, la corrida de M16+M20 con la semilla 20261017 obtiene 60 263 pesos, a 0.088 % del óptimo, y captura el 98.9 % de la mejora. El óptimo **sube los precios premium** entre 4.3 % y 12.8 % y compensa el índice con bajas de hasta 2.9 % en cinco productos de marca propia y estándar. Redondear el óptimo a 10 centavos cuesta 0.007 %; redondearlo a 50 centavos rompe el índice, y hacerlo hacia abajo rompe las existencias. La utilidad es muy plana cerca del óptimo. La corrida mediana de M16+M20 queda a menos de 0.1 % del óptimo en E12-F y en E36-D, y aun así algún precio difiere 2.0 y 4.3 puntos porcentuales del óptimo; en E36-F, a 0.33 %, la diferencia llega a 10.5 puntos.

**Límites.** La demanda es lineal y conocida; los datos son sintéticos; la utilidad es cóncava, de modo que un solucionador de programación cuadrática resuelve cada instancia en 0.11 s o menos, y el algoritmo genético es aquí el objeto de estudio. Las conclusiones describen cuatro instancias y una calibración en una quinta.